# Daltix — Source Discovery & Silver Decisions

**EDA complete for the current scope:** seven source assessments and five relationship checks. This notebook preserves the Raw discovery evidence and the proposals made at that stage. [02 · Silver Pipeline](02_silver_pipeline.ipynb) documents the implemented treatments and final contracts; this notebook does not transform Raw.

**Run locally:** select the project kernel, keep both source switches `False`, then restart and run all. Existing raw files are required; PostgreSQL credentials are not needed.

**Preparation:** [1. Setup](#1-setup--connection) · [2. Source overview](#2-source-table-overview) · [3. Local snapshot](#3-local-raw-extraction) · [4. Profiles](#4-automated-source-profiling) · [5. Reading results](#5-reading-the-results)

**Sources:** [6. Weekly prices](#6-weekly-prices) · [7. Weekly products](#7-weekly-price-products) · [8. Weekly locations](#8-weekly-price-locations) · [9. Prices](#9-prices) · [10. Products](#10-products) · [11. Locations](#11-locations) · [12. Nutritionals](#12-nutritionals)

**Decisions:** [13. Relationships](#13-cross-table-relationships) · [14. Final conclusion](#14-final-eda-conclusion)

Each source and relationship ends with **Established → Caveats → Silver plan (at discovery)**. Detailed findings and delivery steps are in the [README](../README.md#discovery-evidence).


## 1. Setup & Connection

### 1.1 Imports

DuckDB runs SQL; Polars displays results. Source access and snapshot checks use the existing `source_io` helper.


In [1]:
from pathlib import Path

import duckdb
import polars as pl
from dotenv import load_dotenv

from daltix_case.source_io import (
    TABLES,
    extract_snapshot,
    source_config,
    source_connection,
    validate_local_snapshot,
)

### 1.2 Project Configuration

Resolve local paths from the repository root or `notebooks/`. Source discovery and extraction are opt-in; `.env` is loaded only when either switch is enabled.


In [2]:
# Default to local analysis: neither switch enables remote work implicitly.
RUN_SOURCE_DISCOVERY = False
RUN_EXTRACTION = False

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
weekly_prices_path = (RAW_DIR / "weekly_prices.parquet").as_posix()

DB_SCHEMA = None
if RUN_SOURCE_DISCOVERY or RUN_EXTRACTION:
    load_dotenv(PROJECT_ROOT / ".env", override=True)
    DB_SCHEMA = source_config()["schema"]

print(
    "Local analysis mode"
    if not (RUN_SOURCE_DISCOVERY or RUN_EXTRACTION)
    else "Source access enabled"
)

Local analysis mode


### 1.3 Optional PostgreSQL Connection

When enabled, query source identity through a read-only connection that closes automatically. Otherwise skip source access. Timeouts and connection details are documented in the README.


In [3]:
# Query source identity only on explicit opt-in; always close the connection.
connection_info = None
if RUN_SOURCE_DISCOVERY:
    with source_connection() as conn, conn.cursor() as cur:
        cur.execute("""
                SELECT
                    current_database() AS database,
                    current_user AS user,
                    version() AS postgres_version;
            """)
        connection_info = cur.fetchone()
else:
    print("Source connection check skipped.")

Source connection check skipped.


### 1.4 Connection Check Result

`None` is expected in local mode. Review source identity outputs before publishing a notebook run with remote access enabled.


In [4]:
# Connection identity is available only when source discovery was requested.
connection_info

## 2. Source Table Overview

Optional catalog estimates inform extraction planning. Estimated rows are not exact counts; table, index and total sizes measure different storage components. Local counts follow in section 3.3.


In [5]:
# Inspect catalog estimates only when source discovery is explicitly enabled.
source_table_scale = []
if RUN_SOURCE_DISCOVERY:
    with source_connection() as conn, conn.cursor() as cur:
        cur.execute(
            """
                SELECT
                    c.relname AS table_name,
                    c.reltuples::bigint AS estimated_rows,
                    pg_size_pretty(pg_relation_size(c.oid)) AS table_size,
                    pg_size_pretty(pg_indexes_size(c.oid)) AS index_size,
                    pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size
                FROM pg_class AS c
                JOIN pg_namespace AS n ON n.oid = c.relnamespace
                WHERE n.nspname = %s AND c.relkind = 'r'
                ORDER BY c.relname;
                """,
            (DB_SCHEMA,),
        )
        source_table_scale = cur.fetchall()
    source_table_scale = pl.DataFrame(source_table_scale)
else:
    print("Remote source overview skipped; use the local manifest below.")

source_table_scale

Remote source overview skipped; use the local manifest below.


[]

## 3. Local Raw Extraction

Preserve the seven source tables as raw Parquet, then assess them locally. Extraction is explicit, refuses existing files and reconciles exported row counts before publishing a manifest.


### 3.1 Local DuckDB Session

Create one analytical connection. Local analysis needs no PostgreSQL extension; optional extraction uses a separate read-only source connection.


In [6]:
# Local analytical connection; no extension installation or remote attachment.
duck = duckdb.connect()

### 3.2 Optional Source Extraction

Keep `RUN_EXTRACTION = False` to use the existing snapshot. An intentional export requires an unused destination; staging and overwrite protection are described in the README.


In [7]:
# Export is opt-in and refuses to replace any existing snapshot file.
extraction_manifest = extract_snapshot(RAW_DIR, enabled=RUN_EXTRACTION)
print(
    "Extraction completed."
    if extraction_manifest
    else "Extraction skipped; using local raw files."
)

Extraction skipped; using local raw files.


### 3.3 Validate the Local Manifest

Check all seven files against recorded row counts, schemas, sizes and SHA-256 hashes. Missing or changed files stop the run. The historical baseline cannot prove original extraction time or source completeness. Display sizes in MiB.


In [8]:
# Validate all seven files against the local manifest, without source queries.
raw_manifest = validate_local_snapshot(RAW_DIR)
raw_files = [
    {
        "table": entry["table"],
        "rows": entry["rows"],
        "size_mib": round(entry["size_bytes"] / (1024**2), 2),
    }
    for entry in raw_manifest["files"]
]
pl.DataFrame(raw_files)

table,rows,size_mib
str,i64,f64
"""locations""",1638,0.03
"""nutritionals""",1096542,18.84
"""prices""",1198547,3.4
"""products""",32826,3.78
"""weekly_prices""",19218071,619.01
"""weekly_prices_locations""",1230,0.04
"""weekly_prices_products""",114517,7.66


## 4. Automated Source Profiling

Use `SUMMARIZE` for types, ranges, nulls and initial distributions. Cardinalities/quantiles are approximate; SQL null percentages are rounded and exclude semantic missingness. Exact source assessments follow.


In [9]:
# Generate an automated profile for every raw table

profiles = {}

for table in TABLES:
    parquet_path = (RAW_DIR / f"{table}.parquet").as_posix()

    profiles[table] = duck.sql(f"""
        SUMMARIZE
        SELECT *
        FROM read_parquet('{parquet_path}');
    """).pl()

    print(f"Profiled: {table}")

Profiled: locations
Profiled: nutritionals


Profiled: prices
Profiled: products
Profiled: weekly_prices
Profiled: weekly_prices_locations
Profiled: weekly_prices_products


### 4.1 Weekly Prices


In [10]:
profiles["weekly_prices"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""shop""","""VARCHAR""","""frc""","""plc""",4,null,null,null,null,null,19218071,0.00
"""location""","""VARCHAR""","""anderlecht-veeweyde""","""wilrijk""",15,null,null,null,null,null,19218071,0.00
"""daltix_id""","""VARCHAR""","""0000ba625520cd774f3fc738e27d9d…","""fffeb08682566dedbf69cb8c7fd627…",108487,null,null,null,null,null,19218071,0.00
"""week""","""DATE""","""2019-01-07""","""2020-12-28""",93,"""2020-02-04 04:03:35.545879""",null,"""2019-08-21""","""2020-02-13""","""2020-08-01""",19218071,0.00
"""price""","""DOUBLE""","""0.009000000000000001""","""1538.9""",16789,"""6.25093231413216""","""15.446185810047119""","""2.101961706986575""","""3.446990393646293""","""6.148292353247857""",19218071,0.00
"""price_promo""","""DOUBLE""","""0.009000000000000001""","""1538.9""",25480,"""6.060071950857528""","""15.088544886594645""","""2.0050036368562534""","""3.3207493398910057""","""5.960160971633449""",19218071,0.00


### 4.2 Products


In [11]:
profiles["products"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""000225fe97b2c286a59521c3fd7476…","""fffef4e9004a2be76b93557c5ae430…",31353,null,null,null,null,null,32826,0.00
"""shop""","""VARCHAR""","""frc""","""obmuj""",7,null,null,null,null,null,32826,0.00
"""name""","""VARCHAR""","""#N/A""","""Нak Worteltjes 350g""",38389,null,null,null,null,null,32826,0.12
"""brand""","""VARCHAR""","""""","""нема""",3595,null,null,null,null,null,32826,2.25
"""country""","""VARCHAR""","""be""","""nl""",3,null,null,null,null,null,32826,0.00
"""description""","""VARCHAR""","""""","""﻿rundvleessalade met heerlijk …",25674,null,null,null,null,null,32826,10.34
"""categories""","""VARCHAR""","""[ [ ""2+1 op Snoep"" ] ]""","""[ [ ""zuivel, eieren, bot…",11833,null,null,null,null,null,32826,1.51
"""language""","""VARCHAR""","""fr""","""nl""",2,null,null,null,null,null,32826,0.00


### 4.3 Locations


In [12]:
profiles["locations"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""shop""","""VARCHAR""","""frc""","""raps""",10,null,null,null,null,null,1638,0.00
"""country_code""","""VARCHAR""","""be""","""nl""",4,null,null,null,null,null,1638,0.00
"""id""","""VARCHAR""","""0065c""","""ffe44""",1339,null,null,null,null,null,1638,0.00
"""type""","""VARCHAR""",null,null,0,null,null,null,null,null,1638,100.00
"""geolocation_latitude""","""DOUBLE""","""49.5594117""","""59.7139973""",1582,"""50.85913825058744""","""0.39411816770811114""","""50.65655370424243""","""50.87752464216""","""51.09480939058704""",1638,1.10
"""geolocation_longitude""","""DOUBLE""","""2.5924207""","""14.1698445""",1463,"""4.488577900622961""","""0.8197011614008892""","""4.020633977927927""","""4.426998570909091""","""5.069309573242629""",1638,1.10
"""postcode""","""VARCHAR""","""1000""","""9990""",563,null,null,null,null,null,1638,2.44
"""sources""","""VARCHAR""","""[ ""offline"" ]""","""[ ""online"" ]""",3,null,null,null,null,null,1638,0.00


### 4.4 Nutritionals


In [13]:
profiles["nutritionals"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,i32,str,str,str,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""0002c6e437ae0c4f3a9129ba13a702…","""fffffba4674816c2caac0a2e2d913c…",52781,null,null,null,null,null,1096542,0.00
"""shop""","""VARCHAR""","""frc""","""plc""",6,null,null,null,null,null,1096542,0.00
"""country""","""VARCHAR""","""be""","""nl""",2,null,null,null,null,null,1096542,0.00
"""download_date""","""DATE""","""2020-11-27""","""2021-02-24""",90,"""2021-01-01 16:42:41.64415""",null,"""2020-12-11""","""2020-12-26""","""2021-01-28""",1096542,0.00
"""nutritional_values_std""","""VARCHAR""","""{ ""nutrients"": { ""carboh…","""{ ""nutrients"": { ""satura…",46817,null,null,null,null,null,1096542,0.00
"""language""","""VARCHAR""","""nl""","""nl""",1,null,null,null,null,null,1096542,0.00


### 4.5 Prices


In [14]:
profiles["prices"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""03cfa418d809e4ebce7db71eab220b…","""fc0dc30d90155f84a0cce32cfdda94…",121,null,null,null,null,null,1198547,0.00
"""shop""","""VARCHAR""","""frc""","""plc""",5,null,null,null,null,null,1198547,0.00
"""country""","""VARCHAR""","""be""","""nl""",2,null,null,null,null,null,1198547,0.00
"""location""","""VARCHAR""","""0065c""","""ffe44""",113,null,null,null,null,null,1198547,0.00
"""price""","""DOUBLE""","""0.29700000000000004""","""9.779000000000002""",964,"""2.464547105787846""","""1.5071818721092398""","""1.3879432995176715""","""2.0756926887715528""","""3.258620377596155""",1198547,0.00
"""promo_price""","""DOUBLE""","""1.125""","""4.455""",22,"""2.2338778829095007""","""0.6419299210425455""","""1.7009999999999998""","""2.241""","""2.6910000000000003""",1198547,99.44
"""unit_std""","""VARCHAR""","""su""","""su""",1,null,null,null,null,null,1198547,0.00
"""currency""","""VARCHAR""","""eur""","""eur""",1,null,null,null,null,null,1198547,0.00
"""downloaded_on""","""DATE""","""2020-02-25""","""2021-02-25""",371,"""2020-08-26 00:02:44.286924""",null,"""2020-05-29""","""2020-08-26""","""2020-11-23""",1198547,0.00


### 4.6 Weekly Price Locations


In [15]:
profiles["weekly_prices_locations"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""shop""","""VARCHAR""","""frc""","""plc""",4,null,null,null,null,null,1230,0.00
"""location""","""VARCHAR""","""a12wilrijk_sg""","""zwijndrecht_sg""",1084,null,null,null,null,null,1230,0.00
"""location_name""","""VARCHAR""","""'s Gravenwezel""","""op-den-Berg""",1170,null,null,null,null,null,1230,0.00
"""shop_type""","""VARCHAR""","""da""","""yxorp""",10,null,null,null,null,null,1230,35.45
"""geolocation_latitude""","""DOUBLE""","""49.5594117""","""59.7139973""",1212,"""50.8547489389824""","""0.4068419485349983""","""50.66309304960937""","""50.87047912761182""","""51.08206745666667""",1230,0.49
"""geolocation_longitude""","""DOUBLE""","""2.5924516""","""14.1698445""",1126,"""4.500730482690436""","""0.8267645891405758""","""4.041201687890625""","""4.427794845783815""","""5.064025354028514""",1230,0.49
"""locality""","""VARCHAR""","""Aalst""","""Étalle""",448,null,null,null,null,null,1230,0.81
"""postcode""","""VARCHAR""","""1000""","""9990""",560,null,null,null,null,null,1230,1.87
"""state""","""VARCHAR""","""Antwerpen""","""Wallonie""",5,null,null,null,null,null,1230,0.81


### 4.7 Weekly Price Products


In [16]:
profiles["weekly_prices_products"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""0000ba625520cd774f3fc738e27d9d…","""fffff2b33a346e37d69d2e978a5fbb…",105208,null,null,null,null,null,114517,0.00
"""shop""","""VARCHAR""","""frc""","""plc""",4,null,null,null,null,null,114517,0.00
"""name""","""VARCHAR""","""""","""żubrówka bison grass vodka 500…",82958,null,null,null,null,null,114517,0.00
"""brand""","""VARCHAR""","""""","""żubrówka""",7850,null,null,null,null,null,114517,5.77
"""country""","""VARCHAR""","""be""","""be""",1,null,null,null,null,null,114517,0.00
"""description""","""VARCHAR""","""""","""﻿ enkel franstalige versie, ne…",84733,null,null,null,null,null,114517,0.00
"""language""","""VARCHAR""","""nl""","""nl""",1,null,null,null,null,null,114517,0.00
"""categories""","""VARCHAR""","""[ [ ""2+1 op Snoep"" ] ]""","""[ [ ""campaign"", ""Win…",5746,null,null,null,null,null,114517,20.34


## 5. Reading the Results

- **Exact checks override profiles:** weekly prices have 104 weeks and 102,069 IDs; the earlier approximate figures were 93 and 108,487.
- **Missingness depends on the check:** SQL nulls differ from blanks/placeholders. Descriptive fields can remain nullable without rejecting the entity.
- **Grain is meaning; uniqueness is a test.** Equal cardinalities do not establish matching sets, safe joins or universal IDs.
- **Calendar coverage is aggregate:** dates being present does not establish complete product/location histories.
- **Silver is planned:** retain raw evidence, separate observations from assumptions, and record exceptions before applying transformation rules.

Read each table's final conclusion for its Silver plan. The [README](../README.md#discovery-details) retains the detailed profiling, modelling and methodological notes.


## 6. Weekly Prices

### 6.1 Grain & Uniqueness

**Question:** does `(daltix_id, shop, location, week)` identify one weekly product/location observation? Add dimensions progressively and compare distinct combinations with row count.


In [17]:
grain_levels = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT daltix_id) AS unique_daltix_ids,
        COUNT(DISTINCT (daltix_id, shop)) AS unique_daltix_shop,
        COUNT(DISTINCT (daltix_id, shop, location)) AS unique_daltix_shop_location,
        COUNT(DISTINCT (daltix_id, shop, location, week)) AS unique_candidate_grain
    FROM read_parquet('{weekly_prices_path}');
""").pl()

grain_levels

total_rows,unique_daltix_ids,unique_daltix_shop,unique_daltix_shop_location,unique_candidate_grain
i64,i64,i64,i64,i64
19218071,102069,102069,375742,18699187


#### Inspect repeated keys

Split repeated groups into identical `(price, price_promo)` pairs and differing pairs. These counts measure groups; they do not identify a cause or choose a record to keep.


In [18]:
grain_issues = duck.sql(f"""
    WITH grain AS (
        SELECT
            daltix_id,
            shop,
            location,
            week,
            COUNT(*) AS rows_per_grain,
            COUNT(DISTINCT (price, price_promo)) AS distinct_price_pairs
        FROM read_parquet('{weekly_prices_path}')
        GROUP BY ALL
    )

    SELECT
        week,
        COUNT(*) AS duplicated_grain_combinations,
        COUNT(*) FILTER (
            WHERE distinct_price_pairs = 1
        ) AS exact_duplicate_combinations,
        COUNT(*) FILTER (
            WHERE distinct_price_pairs > 1
        ) AS same_grain_different_prices
    FROM grain
    WHERE rows_per_grain > 1
    GROUP BY week
    ORDER BY week;
""").pl()

grain_issues

week,duplicated_grain_combinations,exact_duplicate_combinations,same_grain_different_prices
date,i64,i64,i64
2019-05-27,126619,39196,87423
2019-12-30,188398,47751,140647
2020-12-28,203867,56025,147842


#### Findings — Grain

The working key is not unique: **518,884 repeated groups** occur in three weeks. Of these, **142,972** have identical prices and **375,912** have differing prices.

Rows minus distinct keys also equals 518,884, so each repeated group has two rows: **1,037,768 affected rows (5.40%)**. Adding `shop` does not increase ID cardinality in this snapshot. No cleaning is applied here; see the Silver plan below.


### 6.2 Temporal Coverage

Compare endpoints, distinct dates and expected weekly periods. Calendar alignment is needed to interpret this as continuous weekly coverage.


In [19]:
temporal_coverage = duck.sql(f"""
    SELECT
        MIN(week) AS first_week,
        MAX(week) AS last_week,
        COUNT(DISTINCT week) AS observed_weeks,
        DATE_DIFF('week', MIN(week), MAX(week)) + 1 AS expected_weeks
    FROM read_parquet('{weekly_prices_path}');
""").pl()

temporal_coverage

first_week,last_week,observed_weeks,expected_weeks
date,date,i64,i64
2019-01-07,2020-12-28,104,104


#### Findings — Coverage

**2019-01-07 → 2020-12-28: 104/104 weeks.** An earlier separate local check found zero null weeks and all dates on Mondays; that supplementary check is not an executable cell here. Product/location continuity remains untested.


### 6.3 Missingness

Check identifiers for nulls, blanks and `#N/A`, `N/A`, `NULL`, `NONE`; check date and price fields for SQL nulls.


In [20]:
# Check whether key identifying fields contain NULLs, blank strings
# or common placeholder values that would not appear as missing in basic profiling.

weekly_prices_missingness = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE daltix_id IS NULL
               OR TRIM(daltix_id) = ''
               OR UPPER(TRIM(daltix_id)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        ) AS missing_daltix_id,

        COUNT(*) FILTER (
            WHERE shop IS NULL
               OR TRIM(shop) = ''
               OR UPPER(TRIM(shop)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        ) AS missing_shop,

        COUNT(*) FILTER (
            WHERE location IS NULL
               OR TRIM(location) = ''
               OR UPPER(TRIM(location)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        ) AS missing_location,

        COUNT(*) FILTER (
            WHERE week IS NULL
        ) AS missing_week,

        COUNT(*) FILTER (
            WHERE price IS NULL
        ) AS missing_price,

        COUNT(*) FILTER (
            WHERE price_promo IS NULL
        ) AS missing_price_promo

    FROM read_parquet('{weekly_prices_path}');
""").pl()

weekly_prices_missingness

total_rows,missing_daltix_id,missing_shop,missing_location,missing_week,missing_price,missing_price_promo
i64,i64,i64,i64,i64,i64,i64
19218071,0,0,0,0,0,0


#### Findings — Completeness

No missing values were found by these checks in the identifiers, date or price fields. This establishes tested structural completeness, not business validity.


### 6.4 Promotion Logic

Compare `price_promo` with `price`, overall and by retailer. Non-null availability alone cannot identify a promotion.


In [21]:
# Understand how promotional prices are represented in weekly_prices.
# Compare price_promo with the regular price to identify the main pricing patterns.

promotion_logic = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE price_promo < price
        ) AS promo_below_regular,

        COUNT(*) FILTER (
            WHERE price_promo = price
        ) AS promo_equal_regular,

        COUNT(*) FILTER (
            WHERE price_promo > price
        ) AS promo_above_regular,

        COUNT(*) FILTER (
            WHERE price_promo IS NULL
        ) AS promo_missing

    FROM read_parquet('{weekly_prices_path}');
""").pl()

promotion_logic

total_rows,promo_below_regular,promo_equal_regular,promo_above_regular,promo_missing
i64,i64,i64,i64,i64
19218071,1638468,17568677,10926,0


In [22]:
# Compare promotional price behaviour across retailers.
# This helps determine whether price_promo = price is a systematic encoding pattern.

promotion_logic_by_shop = duck.sql(f"""
    SELECT
        shop,
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE price_promo < price
        ) AS promo_below_regular,

        COUNT(*) FILTER (
            WHERE price_promo = price
        ) AS promo_equal_regular,

        COUNT(*) FILTER (
            WHERE price_promo > price
        ) AS promo_above_regular

    FROM read_parquet('{weekly_prices_path}')
    GROUP BY shop
    ORDER BY shop;
""").pl()

promotion_logic_by_shop

shop,total_rows,promo_below_regular,promo_equal_regular,promo_above_regular
str,i64,i64,i64,i64
"""frc""",12340105,1085356,11246950,7799
"""idla""",168375,1686,166624,65
"""lld""",1610000,91933,1517077,990
"""plc""",5099591,459493,4638026,2072


#### Findings — Promotion

All four shops show the same broad pattern: **17,568,677 equal pairs**, **1,638,468 lower promotional prices**, and **10,926 higher promotional prices**. Equality/no promotion and lower price/promotion are working interpretations, not confirmed source rules. Keep the higher-price cases for investigation.


### 6.5 Price Validity & Outliers

Check non-positive prices and review low/high observations. Thresholds 0.10, 100 and 500 are exploratory, not Silver rejection rules.


In [23]:
# Check basic price validity and quantify extreme values.
# Thresholds are exploratory and are used to identify observations that require inspection.

price_validity = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE price <= 0
        ) AS non_positive_regular_prices,

        COUNT(*) FILTER (
            WHERE price_promo <= 0
        ) AS non_positive_promo_prices,

        COUNT(*) FILTER (
            WHERE price < 0.10
        ) AS regular_price_below_10_cents,

        COUNT(*) FILTER (
            WHERE price > 100
        ) AS regular_price_above_100,

        COUNT(*) FILTER (
            WHERE price > 500
        ) AS regular_price_above_500,

        MIN(price) AS minimum_regular_price,
        MAX(price) AS maximum_regular_price

    FROM read_parquet('{weekly_prices_path}');
""").pl()

price_validity

total_rows,non_positive_regular_prices,non_positive_promo_prices,regular_price_below_10_cents,regular_price_above_100,regular_price_above_500,minimum_regular_price,maximum_regular_price
i64,i64,i64,i64,i64,i64,f64,f64
19218071,0,0,506,38416,3378,0.009,1538.9


In [24]:
# Inspect the highest regular prices to understand whether extreme values
# are concentrated in specific products, retailers, locations or weeks.

highest_prices = duck.sql(f"""
    SELECT
        daltix_id,
        shop,
        location,
        week,
        price,
        price_promo
    FROM read_parquet('{weekly_prices_path}')
    ORDER BY price DESC
    LIMIT 20;
""").pl()

highest_prices

daltix_id,shop,location,week,price,price_promo
str,str,str,date,f64,f64
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""base""",2020-03-30,1538.9,1538.9
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""base""",2020-03-23,1538.9,1538.9
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""mont-st-jean""",2020-05-18,1505.9,1505.9
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""mont-st-jean""",2020-05-11,1505.9,1505.9
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""sint-agatha-berchem""",2020-05-18,1505.9,1505.9
…,…,…,…,…,…
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""base""",2020-05-25,1329.0,1329.0
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""mont-st-jean""",2020-06-15,1329.0,1329.0
"""e2264a18119e1327f49c80e1ef2cf4…","""frc""","""sint-agatha-berchem""",2020-05-25,1329.0,1329.0


In [25]:
# Inspect the lowest positive regular prices.

lowest_prices = duck.sql(f"""
    SELECT
        daltix_id,
        shop,
        location,
        week,
        price,
        price_promo
    FROM read_parquet('{weekly_prices_path}')
    WHERE price > 0
    ORDER BY price
    LIMIT 20;
""").pl()

lowest_prices

daltix_id,shop,location,week,price,price_promo
str,str,str,date,f64,f64
"""ddaa7dd7a107db386ed8b372654412…","""frc""","""base""",2020-07-20,0.009,0.009
"""3f3d5ea5ccc41d85107ae1b08d60ec…","""lld""","""base""",2020-07-20,0.009,0.009
"""213ce7221889a0217ec1e300dc33eb…","""frc""","""base""",2020-10-19,0.009,0.009
"""213ce7221889a0217ec1e300dc33eb…","""frc""","""mont-st-jean""",2020-09-07,0.009,0.009
"""0964f76542c7c71d11af3220e82745…","""lld""","""base""",2020-11-02,0.009,0.009
…,…,…,…,…,…
"""eb149d6a6f11e112a2d7fd3d74d777…","""lld""","""base""",2020-07-13,0.009,0.009
"""303e046f67ac619685aff518d1e632…","""frc""","""wilrijk""",2020-08-10,0.009,0.009
"""213ce7221889a0217ec1e300dc33eb…","""frc""","""base""",2020-10-12,0.009,0.009


#### Findings — Validity

No non-positive prices. Regular prices range **0.009–1,538.90**: 506 rows below 0.10, 38,416 above 100, including 3,378 above 500. Selected extremes recur across weeks/locations; recurrence does not rule out systematic errors. Retain them for product-context validation.


### 6.6 Conclusion — Weekly Prices

**Established:** 19,218,071 rows; working grain `(daltix_id, shop, location, week)`; **104/104 weeks** and no missing core values under the checks used.

**Caveats:** three weeks contain **142,972 identical-price duplicate groups** and **375,912 differing-price groups**. Equal prices dominate; **10,926** rows have `price_promo > price`. Extremes recur, but their commercial validity is unconfirmed.

**Silver plan at discovery:** remove only full-row exact duplicates; preserve and flag conflicting versions at the working grain. Document the promotion rule before deriving status; flag higher promotional prices for review. Retain extreme prices until product context supports a correction. No transformation has been applied here.


## 7. Weekly Price Products

### 7.1 Grain & Uniqueness

Test `daltix_id`, then add shop/country context. Source documentation describes the identifier in retailer/country context; do not assume universal product matching.


In [26]:
# Understand the level of uniqueness in the product dataset.
# The goal is to see whether daltix_id alone is enough to identify a product,
# or whether shop and country are also required.

weekly_products_path = (RAW_DIR / "weekly_prices_products.parquet").as_posix()

product_grain_levels = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT daltix_id) AS unique_daltix_ids,
        COUNT(DISTINCT (daltix_id, shop)) AS unique_daltix_shop,
        COUNT(DISTINCT (daltix_id, shop, country)) AS unique_product_context
    FROM read_parquet('{weekly_products_path}');
""").pl()

product_grain_levels

total_rows,unique_daltix_ids,unique_daltix_shop,unique_product_context
i64,i64,i64,i64
114517,114517,114517,114517


#### Findings — Grain

**114,517 rows = 114,517 distinct IDs.** `daltix_id` is the candidate business key in this snapshot; adding shop/country creates no extra combinations.


### 7.2 Missingness

Count SQL nulls, trimmed blanks and the listed placeholder tokens in descriptive attributes.


In [27]:
# Check product attributes for both technical and semantic missingness.
# Besides SQL NULLs, blank strings and common placeholder values are treated as missing.

product_missingness = duck.sql(f"""
    SELECT
        'name' AS field,
        COUNT(*) FILTER (
            WHERE name IS NULL
               OR TRIM(name) = ''
               OR UPPER(TRIM(name)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        ) AS missing_rows
    FROM read_parquet('{weekly_products_path}')

    UNION ALL

    SELECT
        'brand',
        COUNT(*) FILTER (
            WHERE brand IS NULL
               OR TRIM(brand) = ''
               OR UPPER(TRIM(brand)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        )
    FROM read_parquet('{weekly_products_path}')

    UNION ALL

    SELECT
        'description',
        COUNT(*) FILTER (
            WHERE description IS NULL
               OR TRIM(description) = ''
               OR UPPER(TRIM(description)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        )
    FROM read_parquet('{weekly_products_path}')

    UNION ALL

    SELECT
        'categories',
        COUNT(*) FILTER (
            WHERE categories IS NULL
               OR TRIM(categories) = ''
               OR UPPER(TRIM(categories)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        )
    FROM read_parquet('{weekly_products_path}')
""").pl()

product_missingness = product_missingness.with_columns(
    (pl.col("missing_rows") / product_grain_levels.item(0, "total_rows") * 100)
    .round(2)
    .alias("missing_pct")
)

product_missingness

field,missing_rows,missing_pct
str,i64,f64
"""name""",9,0.01
"""brand""",9589,8.37
"""description""",14562,12.72
"""categories""",23294,20.34


#### Findings — Missingness

Name: **9 rows (0.01%)**; brand: **8.37%**; description: **12.72%**; categories: **20.34%**. Missing descriptive attributes do not invalidate product identity.


### 7.3 Attribute Consistency

Check country/language cardinality per retailer. `MIN` displays one observed value; it is representative only when the distinct count is one.


In [28]:
# Check whether each retailer is associated with a consistent country and language.
# This helps identify unexpected structural inconsistencies in the product dataset.

product_attribute_consistency = duck.sql(f"""
    SELECT
        shop,
        COUNT(*) AS product_rows,
        COUNT(DISTINCT country) AS distinct_countries,
        COUNT(DISTINCT language) AS distinct_languages,
        MIN(country) AS country,
        MIN(language) AS language
    FROM read_parquet('{weekly_products_path}')
    GROUP BY shop
    ORDER BY shop;
""").pl()

product_attribute_consistency

shop,product_rows,distinct_countries,distinct_languages,country,language
str,i64,i64,i64,str,str
"""frc""",44391,1,1,"""be""","""nl"""
"""idla""",13166,1,1,"""be""","""nl"""
"""lld""",33369,1,1,"""be""","""nl"""
"""plc""",23591,1,1,"""be""","""nl"""


#### Findings — Context

All four retailers have `country = be` and `language = nl` in the current product snapshot. These are separate country and language codes, not two countries.


### 7.4 Conclusion — Weekly Price Products

**Established:** 114,517 product records; `daltix_id` is unique. All current rows are Belgian (`be`) with Dutch (`nl`) product text.

**Caveats:** semantic missingness in name **0.01%**, brand **8.37%**, description **12.72%**, categories **20.34%**. Cross-table identity and enrichment coverage are not established by uniqueness alone.

**Silver plan at discovery:** use `daltix_id` as the source business key; normalize the assessed blank/placeholder values to `NULL`; retain incomplete attributes as nullable. Keep shop/country/language. Test `products` as a fallback before backfilling brand, description or categories.


## 8. Weekly Price Locations

### 8.1 Grain & Uniqueness

Test the `location` identifier alone and within `shop`. `location_name` is a separate descriptive field.


In [29]:
# Determine whether the source location identifier is unique on its own
# or only unique within a specific retailer.

weekly_locations_path = (RAW_DIR / "weekly_prices_locations.parquet").as_posix()

location_identity = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT location) AS unique_location_names,
        COUNT(DISTINCT (shop, location)) AS unique_shop_location_pairs
    FROM read_parquet('{weekly_locations_path}');
""").pl()

location_identity

total_rows,unique_location_names,unique_shop_location_pairs
i64,i64,i64
1230,1140,1230


### 8.2 Missingness

Measure null/blank descriptive attributes and missing coordinates. Percentages describe source records, not validated physical stores.


In [30]:
# Check how complete the main location attributes are.
# The goal is to distinguish strong identifying fields from optional attributes
# that may contain missing information.

location_missingness = duck.sql(f"""
    SELECT
        'location_name' AS field,
        COUNT(*) FILTER (
            WHERE location_name IS NULL
               OR TRIM(location_name) = ''
        ) AS missing_rows

    FROM read_parquet('{weekly_locations_path}')

    UNION ALL

    SELECT
        'shop_type',
        COUNT(*) FILTER (
            WHERE shop_type IS NULL
               OR TRIM(shop_type) = ''
        )
    FROM read_parquet('{weekly_locations_path}')

    UNION ALL

    SELECT
        'latitude',
        COUNT(*) FILTER (
            WHERE geolocation_latitude IS NULL
        )
    FROM read_parquet('{weekly_locations_path}')

    UNION ALL

    SELECT
        'longitude',
        COUNT(*) FILTER (
            WHERE geolocation_longitude IS NULL
        )
    FROM read_parquet('{weekly_locations_path}')

    UNION ALL

    SELECT
        'locality',
        COUNT(*) FILTER (
            WHERE locality IS NULL
               OR TRIM(locality) = ''
        )
    FROM read_parquet('{weekly_locations_path}')

    UNION ALL

    SELECT
        'postcode',
        COUNT(*) FILTER (
            WHERE postcode IS NULL
               OR TRIM(postcode) = ''
        )
    FROM read_parquet('{weekly_locations_path}')

    UNION ALL

    SELECT
        'state',
        COUNT(*) FILTER (
            WHERE state IS NULL
               OR TRIM(state) = ''
        )
    FROM read_parquet('{weekly_locations_path}');
""").pl()

location_missingness = location_missingness.with_columns(
    (pl.col("missing_rows") / location_identity.item(0, "total_rows") * 100)
    .round(2)
    .alias("missing_pct")
)

location_missingness.sort("missing_pct", descending=True)

field,missing_rows,missing_pct
str,i64,f64
"""shop_type""",436,35.45
"""postcode""",23,1.87
"""locality""",10,0.81
"""state""",10,0.81
"""latitude""",6,0.49
"""longitude""",6,0.49
"""location_name""",0,0.0


#### Findings — Missingness

`shop_type`: **35.45%**; postcode: **1.87%**; locality/state: **0.81%** each; latitude/longitude: **0.49%** each. `location_name` is populated. Keep optional attributes nullable.


### 8.3 Attribute Consistency

Inspect reused names, shared coordinates and retailer-specific `shop_type` patterns. Shared descriptions do not establish duplicate entities.


In [31]:
# Check whether the same location name is associated with multiple location identifiers within the same retailer.

location_name_consistency = duck.sql(f"""
    SELECT
        shop,
        location_name,
        COUNT(DISTINCT location) AS different_location_ids
    FROM read_parquet('{weekly_locations_path}')
    GROUP BY shop, location_name
    HAVING COUNT(DISTINCT location) > 1
    ORDER BY different_location_ids DESC, shop, location_name;
""").pl()

location_name_consistency

shop,location_name,different_location_ids
str,str,i64
"""idla""","""Default""",4
"""lld""","""Bastogne""",2
"""lld""","""Denderleeuw""",2
"""lld""","""Dilbeek""",2
"""lld""","""Eeklo""",2
…,…,…
"""lld""","""Merksem""",2
"""lld""","""Overijse""",2
"""lld""","""Rixensart""",2


In [32]:
# Inspect location records that share the same displayed location name.
# This helps determine whether they represent distinct physical locations
# or potentially duplicated/ambiguous records.

reused_location_details = duck.sql(f"""
    WITH reused_names AS (
        SELECT
            shop,
            location_name
        FROM read_parquet('{weekly_locations_path}')
        GROUP BY shop, location_name
        HAVING COUNT(DISTINCT location) > 1
    )

    SELECT
        l.shop,
        l.location,
        l.location_name,
        l.shop_type,
        l.locality,
        l.postcode,
        l.state,
        l.geolocation_latitude AS latitude,
        l.geolocation_longitude AS longitude
    FROM read_parquet('{weekly_locations_path}') l

    INNER JOIN reused_names r
        ON l.shop = r.shop
       AND l.location_name = r.location_name

    ORDER BY l.shop, l.location_name, l.location;
""").pl()

reused_location_details

shop,location,location_name,shop_type,locality,postcode,state,latitude,longitude
str,str,str,str,str,str,str,f64,f64
"""idla""","""base""","""Default""",null,null,null,null,null,null
"""idla""","""de""","""Default""",null,null,null,null,null,null
"""idla""","""lu""","""Default""",null,null,null,null,null,null
"""idla""","""nl""","""Default""",null,null,null,null,null,null
"""lld""","""bastogne_ad""","""Bastogne""","""da""","""Bastogne""","""6600""","""Wallonie""",50.002625,5.6901511
…,…,…,…,…,…,…,…,…
"""lld""","""rixensart_sg""","""Rixensart""","""og & pohs""","""Rixensart""","""1330""","""Wallonie""",50.711012,4.519522
"""lld""","""strombeek_pr""","""Strombeek""","""yxorp""","""Grimbergen""","""1853""","""Vlaanderen""",50.908571,4.3585046
"""lld""","""strombeek_sg""","""Strombeek""","""og & pohs""","""Grimbergen""","""1853""","""Vlaanderen""",50.904345,4.3626549


#### Findings — Name

`Default` and several city names are reused for distinct `location` IDs. Keep `location_name` descriptive; the candidate key remains `(shop, location)`.


In [33]:
# Check whether multiple location records share exactly the same coordinates.
# This can reveal possible duplicate physical locations or shared geocoding.

shared_coordinates = duck.sql(f"""
    SELECT
        geolocation_latitude AS latitude,
        geolocation_longitude AS longitude,
        COUNT(*) AS location_records,
        COUNT(DISTINCT (shop, location)) AS different_locations
    FROM read_parquet('{weekly_locations_path}')
    WHERE geolocation_latitude IS NOT NULL
      AND geolocation_longitude IS NOT NULL
    GROUP BY geolocation_latitude, geolocation_longitude
    HAVING COUNT(DISTINCT (shop, location)) > 1
    ORDER BY different_locations DESC;
""").pl()

shared_coordinates

latitude,longitude,location_records,different_locations
f64,f64,i64,i64
51.203018,4.5701976,2,2
50.821764,5.1791706,2,2
51.067271,3.6868082,2,2
51.025875,3.7723651,2,2
50.849109,4.3535698,2,2
50.54854,5.946391,2,2
49.559417,5.840447,2,2


In [34]:
# Inspect location records that share exactly the same coordinates.
# This helps determine whether they are legitimate co-located locations
# or possible duplicate/alias records.

shared_coordinate_details = duck.sql(f"""
    WITH shared_coords AS (
        SELECT
            geolocation_latitude,
            geolocation_longitude
        FROM read_parquet('{weekly_locations_path}')
        WHERE geolocation_latitude IS NOT NULL
          AND geolocation_longitude IS NOT NULL
        GROUP BY
            geolocation_latitude,
            geolocation_longitude
        HAVING COUNT(DISTINCT (shop, location)) > 1
    )

    SELECT
        l.shop,
        l.location,
        l.location_name,
        l.shop_type,
        l.locality,
        l.postcode,
        l.geolocation_latitude AS latitude,
        l.geolocation_longitude AS longitude
    FROM read_parquet('{weekly_locations_path}') l

    INNER JOIN shared_coords s
        ON l.geolocation_latitude = s.geolocation_latitude
       AND l.geolocation_longitude = s.geolocation_longitude

    ORDER BY latitude, longitude, shop, location;
""").pl()

shared_coordinate_details

shop,location,location_name,shop_type,locality,postcode,latitude,longitude
str,str,str,str,str,str,f64,f64
"""idla""","""athus""","""Athus""",null,"""Aubange""","""6791""",49.559417,5.840447
"""lld""","""athus_ad""","""Athus""","""da""","""Aubange""","""6791""",49.559417,5.840447
"""lld""","""jalhay_pr""","""Jalhay""","""yxorp""","""Jalhay""","""4845""",50.54854,5.946391
"""lld""","""tiege_pr""","""Tiege""","""yxorp""","""Jalhay""","""4845""",50.54854,5.946391
"""lld""","""landen_ad""","""Landen""","""da""","""Sint-Truiden""","""3800""",50.821764,5.1791706
…,…,…,…,…,…,…,…
"""idla""","""melle""","""Melle""",null,"""Melle""","""9050""",51.025875,3.7723651
"""idla""","""gent3""","""Gent Brugsesteenweg 265/e""",null,"""Gent""","""9030""",51.067271,3.6868082
"""lld""","""mariakerke_pr""","""Mariakerke""","""yxorp""","""Gent""","""9030""",51.067271,3.6868082


In [35]:
# Assess shop_type completeness and diversity by retailer.
# This shows whether missing values are systematic for particular shops.

shop_type_by_retailer = duck.sql(f"""
    SELECT
        shop,
        COUNT(*) AS location_rows,

        COUNT(*) FILTER (
            WHERE shop_type IS NULL
               OR TRIM(shop_type) = ''
        ) AS missing_shop_type,

        COUNT(DISTINCT shop_type) AS different_shop_types

    FROM read_parquet('{weekly_locations_path}')
    GROUP BY shop
    ORDER BY shop;
""").pl()

shop_type_by_retailer

shop,location_rows,missing_shop_type,different_shop_types
str,i64,i64,i64
"""frc""",47,1,4
"""idla""",392,392,0
"""lld""",749,1,5
"""plc""",42,42,0


In [36]:
# Separate shared coordinates within the same retailer from coordinates
# shared across different retailers.

shared_coordinate_patterns = duck.sql(f"""
    SELECT
        geolocation_latitude AS latitude,
        geolocation_longitude AS longitude,
        COUNT(*) AS location_records,
        COUNT(DISTINCT shop) AS different_shops,
        COUNT(DISTINCT (shop, location)) AS different_locations
    FROM read_parquet('{weekly_locations_path}')
    WHERE geolocation_latitude IS NOT NULL
      AND geolocation_longitude IS NOT NULL
    GROUP BY
        geolocation_latitude,
        geolocation_longitude
    HAVING COUNT(DISTINCT (shop, location)) > 1
    ORDER BY different_shops DESC, different_locations DESC;
""").pl()

shared_coordinate_patterns

latitude,longitude,location_records,different_shops,different_locations
f64,f64,i64,i64,i64
49.559417,5.840447,2,2,2
51.067271,3.6868082,2,2,2
51.203018,4.5701976,2,1,2
50.54854,5.946391,2,1,2
50.821764,5.1791706,2,1,2
51.025875,3.7723651,2,1,2
50.849109,4.3535698,2,1,2


#### Findings — Coordinate

**Seven coordinate pairs** are shared: two across retailers and five within one retailer. Co-location, aliases or shared geocoding are possible; equal coordinates alone do not justify deduplication.


In [37]:
# Check whether shop_type missingness is concentrated in specific retailers.

shop_type_by_retailer = duck.sql(f"""
    SELECT
        shop,
        COUNT(*) AS total_locations,

        COUNT(*) FILTER (
            WHERE shop_type IS NULL
               OR TRIM(shop_type) = ''
        ) AS missing_shop_type,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE shop_type IS NULL
                   OR TRIM(shop_type) = ''
            ) / COUNT(*),
            2
        ) AS missing_shop_type_pct,

        COUNT(DISTINCT shop_type) AS distinct_shop_types

    FROM read_parquet('{weekly_locations_path}')
    GROUP BY shop
    ORDER BY missing_shop_type_pct DESC;
""").pl()

shop_type_by_retailer

shop,total_locations,missing_shop_type,missing_shop_type_pct,distinct_shop_types
str,i64,i64,f64,i64
"""idla""",392,392,100.0,0
"""plc""",42,42,100.0,0
"""frc""",47,1,2.13,4
"""lld""",749,1,0.13,5


#### Findings — Shop-Type

`idla` and `plc`: **100% missing**. `frc`: **2.13%**; `lld`: **0.13%**. The pattern is retailer-specific; the reason is unconfirmed.


### 8.4 Geographic Plausibility

Check coordinate bounds, incomplete pairs, postcode formats and observed extremes. A four-digit postcode is a diagnostic pattern, not a universal country rule.


In [38]:
# Check technical coordinate validity and describe postcode formatting.
# A four-digit postcode is treated as an observed format pattern,
# not yet as a business validity rule.

geographic_validity = duck.sql(f"""
    SELECT
        COUNT(*) AS total_locations,

        COUNT(*) FILTER (
            WHERE geolocation_latitude IS NOT NULL
              AND (geolocation_latitude < -90 OR geolocation_latitude > 90)
        ) AS invalid_latitudes,

        COUNT(*) FILTER (
            WHERE geolocation_longitude IS NOT NULL
              AND (geolocation_longitude < -180 OR geolocation_longitude > 180)
        ) AS invalid_longitudes,

        COUNT(*) FILTER (
            WHERE
                (geolocation_latitude IS NULL AND geolocation_longitude IS NOT NULL)
                OR
                (geolocation_latitude IS NOT NULL AND geolocation_longitude IS NULL)
        ) AS incomplete_coordinate_pairs,

        COUNT(*) FILTER (
            WHERE postcode IS NOT NULL
              AND NOT regexp_full_match(TRIM(postcode), '[0-9]{{4}}')
        ) AS non_4_digit_postcodes,

        MIN(geolocation_latitude) AS minimum_latitude,
        MAX(geolocation_latitude) AS maximum_latitude,
        MIN(geolocation_longitude) AS minimum_longitude,
        MAX(geolocation_longitude) AS maximum_longitude

    FROM read_parquet('{weekly_locations_path}');
""").pl()

geographic_validity

total_locations,invalid_latitudes,invalid_longitudes,incomplete_coordinate_pairs,non_4_digit_postcodes,minimum_latitude,maximum_latitude,minimum_longitude,maximum_longitude
i64,i64,i64,i64,i64,f64,f64,f64,f64
1230,0,0,0,4,49.559412,59.713997,2.5924516,14.169844


In [39]:
# Inspect geographic extremes to understand whether unusual coordinates
# belong to legitimate locations or indicate source/geocoding issues.

geographic_extremes = duck.sql(f"""
    SELECT
        shop,
        location,
        location_name,
        locality,
        postcode,
        state,
        geolocation_latitude AS latitude,
        geolocation_longitude AS longitude
    FROM read_parquet('{weekly_locations_path}')
    WHERE geolocation_latitude = (
              SELECT MAX(geolocation_latitude)
              FROM read_parquet('{weekly_locations_path}')
          )
       OR geolocation_latitude = (
              SELECT MIN(geolocation_latitude)
              FROM read_parquet('{weekly_locations_path}')
          )
       OR geolocation_longitude = (
              SELECT MAX(geolocation_longitude)
              FROM read_parquet('{weekly_locations_path}')
          )
       OR geolocation_longitude = (
              SELECT MIN(geolocation_longitude)
              FROM read_parquet('{weekly_locations_path}')
          )
    ORDER BY latitude, longitude;
""").pl()

geographic_extremes

shop,location,location_name,locality,postcode,state,latitude,longitude
str,str,str,str,str,str,f64,f64
"""lld""","""halanzy_ad""","""Halanzy""","""Aubange""","""6792""","""Wallonie""",49.559412,5.7297836
"""lld""","""depanne_pr""","""De Panne""","""De Panne""","""8660""","""Vlaanderen""",51.100127,2.5924516
"""idla""","""filipstad""","""Filipstad""","""Filipstad""","""68234""","""Värmlands län""",59.713997,14.169844


#### Findings — Geography

No out-of-range coordinates or half-missing pairs; **four** populated postcodes do not match four digits. The earlier review described extremes as plausible Belgian/Swedish locations; the saved queries establish bounds, not address correctness. Retain the records and avoid one country-specific postcode rule.


### 8.5 Conclusion — Weekly Price Locations

**Established:** 1,230 rows and unique `(shop, location)` pairs; `location` alone is not unique. Populated coordinates pass range/pair checks.

**Caveats:** `shop_type` is **35.45% missing**, including all `idla`/`plc` records. Names are descriptive. Seven coordinate pairs are shared, five within a retailer; geographic extremes and postcode formats need country context.

**Silver plan at discovery:** use `(shop, location)` as business key; keep names descriptive and `shop_type` nullable. Preserve geographic extremes and incomplete optional attributes; do not merge records solely on shared names or coordinates.


## 9. Prices

### 9.1 Grain & Uniqueness

Test `(daltix_id, shop, location, downloaded_on)` as the identifying combination for a dated product/location price observation.


In [40]:
# Understand how row uniqueness changes as retailer, location and date context are added.
# This helps determine the likely grain of the historical price dataset.

prices_path = (RAW_DIR / "prices.parquet").as_posix()

prices_grain = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT daltix_id)
            AS unique_daltix_ids,

        COUNT(DISTINCT (daltix_id, shop))
            AS unique_product_shop_pairs,

        COUNT(DISTINCT (daltix_id, shop, location))
            AS unique_product_location_pairs,

        COUNT(DISTINCT (daltix_id, shop, location, downloaded_on))
            AS unique_candidate_grain

    FROM read_parquet('{prices_path}');
""").pl()

prices_grain

total_rows,unique_daltix_ids,unique_product_shop_pairs,unique_product_location_pairs,unique_candidate_grain
i64,i64,i64,i64,i64
1198547,116,116,3631,1198547


#### Findings — Grain

**1,198,547 rows = 1,198,547 candidate keys.** There are 116 distinct products and 3,631 product/shop/location combinations. `shop` adds no ID cardinality in this sample; location and date add detail.


### 9.2 Missingness

Check core fields for SQL nulls and text blanks. Report `promo_price` separately because its absence requires semantic interpretation.


In [41]:
# Check missing values across the historical price dataset.
# promo_price is kept separate because NULL may represent "no promotion" rather than poor data quality.

prices_missingness = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE daltix_id IS NULL
               OR TRIM(daltix_id) = ''
        ) AS missing_daltix_id,

        COUNT(*) FILTER (
            WHERE shop IS NULL
               OR TRIM(shop) = ''
        ) AS missing_shop,

        COUNT(*) FILTER (
            WHERE country IS NULL
               OR TRIM(country) = ''
        ) AS missing_country,

        COUNT(*) FILTER (
            WHERE location IS NULL
               OR TRIM(location) = ''
        ) AS missing_location,

        COUNT(*) FILTER (
            WHERE price IS NULL
        ) AS missing_price,

        COUNT(*) FILTER (
            WHERE promo_price IS NULL
        ) AS missing_promo_price,

        COUNT(*) FILTER (
            WHERE unit_std IS NULL
               OR TRIM(unit_std) = ''
        ) AS missing_unit_std,

        COUNT(*) FILTER (
            WHERE currency IS NULL
               OR TRIM(currency) = ''
        ) AS missing_currency,

        COUNT(*) FILTER (
            WHERE downloaded_on IS NULL
        ) AS missing_downloaded_on

    FROM read_parquet('{prices_path}');
""").pl()

prices_missingness

total_rows,missing_daltix_id,missing_shop,missing_country,missing_location,missing_price,missing_promo_price,missing_unit_std,missing_currency,missing_downloaded_on
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1198547,0,0,0,0,0,1191783,0,0,0


#### Findings — Missingness

No missing core values under the checks used. `promo_price` is null in **1,191,783 rows (99.44%)**; its meaning is assessed below.


### 9.3 Temporal Coverage

Check dataset-level dates without assuming every product/location appears daily.


In [42]:
# Check the overall time range and whether all calendar dates
# between the first and last observation are represented in the dataset.

prices_temporal_coverage = duck.sql(f"""
    SELECT
        MIN(downloaded_on) AS first_date,
        MAX(downloaded_on) AS last_date,
        COUNT(DISTINCT downloaded_on) AS observed_dates,
        DATE_DIFF(
            'day',
            MIN(downloaded_on),
            MAX(downloaded_on)
        ) + 1 AS calendar_days

    FROM read_parquet('{prices_path}');
""").pl()

prices_temporal_coverage

first_date,last_date,observed_dates,calendar_days
date,date,i64,i64
2020-02-25,2021-02-25,367,367


#### Findings — Coverage

**2020-02-25 → 2021-02-25: 367/367 dates.** No calendar date is absent from the sample; individual series may still have gaps.


### 9.4 Promotion Logic

Compare populated `promo_price` values with `price`. This tests observed price relationships.


In [43]:
# Check how populated promotional prices relate to the regular price.
# This helps determine whether promo_price behaves as an optional active-promotion field.

prices_promotion_logic = duck.sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE promo_price IS NOT NULL
        ) AS populated_promo_rows,

        COUNT(*) FILTER (
            WHERE promo_price < price
        ) AS promo_below_regular,

        COUNT(*) FILTER (
            WHERE promo_price = price
        ) AS promo_equal_regular,

        COUNT(*) FILTER (
            WHERE promo_price > price
        ) AS promo_above_regular

    FROM read_parquet('{prices_path}');
""").pl()

prices_promotion_logic

populated_promo_rows,promo_below_regular,promo_equal_regular,promo_above_regular
i64,i64,i64,i64
6764,6764,0,0


#### Findings — Promotion

All **6,764 populated promotional prices are lower** than regular price; none is equal or higher. This is consistent with an optional promotional field. It does not prove that the null rows had no promotion: record them as **promotion not recorded; source meaning unconfirmed**.


### 9.5 Unit & Currency Consistency

Check observed unit and currency codes. Matching codes alone do not establish comparable pack sizes or equivalent products.


In [44]:
# Check whether standardized unit and currency values are consistent
# across the historical price sample.

price_unit_currency = duck.sql(f"""
    SELECT
        COUNT(DISTINCT unit_std) AS distinct_units,
        COUNT(DISTINCT currency) AS distinct_currencies,
        MIN(unit_std) AS unit_std,
        MIN(currency) AS currency
    FROM read_parquet('{prices_path}');
""").pl()

price_unit_currency

distinct_units,distinct_currencies,unit_std,currency
i64,i64,str,str
1,1,"""su""","""eur"""


#### Findings — Unit/Currency

All rows use `unit_std = su` and `currency = eur`. Retain both fields explicitly; product/quantity comparability requires separate validation.


### 9.6 Price Validity & Outliers

Check non-positive prices and exploratory extremes. The checks do not define product-specific plausibility.


In [45]:
# Check for non-positive and extreme price values in the historical price sample.
# The thresholds are exploratory and used only to identify observations for review.

prices_validity = duck.sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE price <= 0
        ) AS non_positive_regular_prices,

        COUNT(*) FILTER (
            WHERE promo_price IS NOT NULL
              AND promo_price <= 0
        ) AS non_positive_promo_prices,

        MIN(price) AS minimum_regular_price,
        MAX(price) AS maximum_regular_price,

        COUNT(*) FILTER (
            WHERE price < 0.10
        ) AS regular_prices_below_10_cents,

        COUNT(*) FILTER (
            WHERE price > 100
        ) AS regular_prices_above_100

    FROM read_parquet('{prices_path}');
""").pl()

prices_validity

non_positive_regular_prices,non_positive_promo_prices,minimum_regular_price,maximum_regular_price,regular_prices_below_10_cents,regular_prices_above_100
i64,i64,f64,f64,i64,i64
0,0,0.297,9.779,0,0


#### Findings — Validity

No non-positive prices; regular prices span **€0.297–€9.779**, with none below €0.10 or above €100. No filtering is justified by these checks.


### 9.7 Conclusion — Prices

**Established:** 1,198,547 rows; unique `(daltix_id, shop, location, downloaded_on)`; **367/367 dates**. Unit/currency are `su`/`eur`; regular prices span **€0.297–€9.779** and pass the checks used.

**Caveats:** `promo_price` is **99.44% null**. All **6,764 populated values** are lower, but null does not by itself prove absence of promotion. Calendar completeness is aggregate, not per product/location.

**Silver plan at discovery:** retain the full business key, unit and currency; no price filtering is currently justified. Preserve null promotional values. Define promotion status only after documenting their meaning; do not automatically convert unknown promotion status to false.


**Implemented follow-up:** see [02 · Silver Pipeline](02_silver_pipeline.ipynb) for the adopted promotion convention, deterministic nutritional selection and persisted provenance. These decisions extend the discovery evidence; Raw results below remain unchanged.


## 10. Products

### 10.1 Grain & Uniqueness

Test `daltix_id`, then add retailer/country context.


In [46]:
# Check how product uniqueness changes when shop and country context are added.
# This helps determine the most appropriate grain and candidate key for the table.

products_path = (RAW_DIR / "products.parquet").as_posix()

products_grain = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT daltix_id)
            AS unique_daltix_ids,

        COUNT(DISTINCT (daltix_id, shop))
            AS unique_product_shop_pairs,

        COUNT(DISTINCT (daltix_id, shop, country))
            AS unique_product_contexts

    FROM read_parquet('{products_path}');
""").pl()

products_grain

total_rows,unique_daltix_ids,unique_product_shop_pairs,unique_product_contexts
i64,i64,i64,i64
32826,32826,32826,32826


#### Findings — Grain

**32,826 rows = 32,826 distinct IDs.** Adding shop/country creates no extra combinations; use `daltix_id` as the candidate source business key.


### 10.2 Missingness

Count nulls, trimmed blanks and the listed placeholder tokens in product descriptions.


In [47]:
# Check semantic missingness across the main descriptive product attributes.
# Missing values include SQL NULLs, blank strings and common placeholder values.

products_missingness = duck.sql(f"""
    SELECT
        'name' AS field,
        COUNT(*) FILTER (
            WHERE name IS NULL
               OR TRIM(name) = ''
               OR UPPER(TRIM(name)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        ) AS missing_rows

    FROM read_parquet('{products_path}')

    UNION ALL

    SELECT
        'brand',
        COUNT(*) FILTER (
            WHERE brand IS NULL
               OR TRIM(brand) = ''
               OR UPPER(TRIM(brand)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        )
    FROM read_parquet('{products_path}')

    UNION ALL

    SELECT
        'description',
        COUNT(*) FILTER (
            WHERE description IS NULL
               OR TRIM(description) = ''
               OR UPPER(TRIM(description)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        )
    FROM read_parquet('{products_path}')

    UNION ALL

    SELECT
        'categories',
        COUNT(*) FILTER (
            WHERE categories IS NULL
               OR TRIM(categories) = ''
               OR UPPER(TRIM(categories)) IN ('#N/A', 'N/A', 'NULL', 'NONE')
        )
    FROM read_parquet('{products_path}')
""").pl()

products_missingness = products_missingness.with_columns(
    (pl.col("missing_rows") / products_grain.item(0, "total_rows") * 100)
    .round(2)
    .alias("missing_pct")
)

products_missingness.sort("missing_pct", descending=True)

field,missing_rows,missing_pct
str,i64,f64
"""description""",8013,24.41
"""brand""",1279,3.9
"""categories""",497,1.51
"""name""",42,0.13


#### Findings — Missingness

Description: **24.41%**; brand: **3.90%**; categories: **1.51%**; name: **0.13%**. Treat these as nullable attributes, not required identifiers.


### 10.3 Market & Language Consistency

Measure country/language cardinality by retailer. When counts exceed one, `MIN` is only an example value.


In [48]:
# Check whether each retailer is associated with a consistent country and language.
# This helps identify unexpected structural inconsistencies in the product sample.

products_attribute_consistency = duck.sql(f"""
    SELECT
        shop,
        COUNT(*) AS product_rows,
        COUNT(DISTINCT country) AS distinct_countries,
        COUNT(DISTINCT language) AS distinct_languages,
        MIN(country) AS country,
        MIN(language) AS language
    FROM read_parquet('{products_path}')
    GROUP BY shop
    ORDER BY shop;
""").pl()

products_attribute_consistency

shop,product_rows,distinct_countries,distinct_languages,country,language
str,i64,i64,i64,str,str
"""frc""",6327,1,1,"""be""","""nl"""
"""gc""",5803,2,1,"""be""","""nl"""
"""ha""",9889,2,1,"""be""","""nl"""
"""idla""",1345,3,2,"""be""","""fr"""
"""ldil""",226,2,1,"""be""","""nl"""
"""lld""",3900,1,1,"""be""","""nl"""
"""obmuj""",5336,1,1,"""nl""","""nl"""


#### Inspect observed combinations

List actual shop/country/language combinations before interpreting multi-market coverage.


In [49]:
# Inspect the actual country and language combinations used by each retailer.
# This shows whether multi-country or multi-language patterns are systematic.

product_market_combinations = duck.sql(f"""
    SELECT
        shop,
        country,
        language,
        COUNT(*) AS product_rows
    FROM read_parquet('{products_path}')
    GROUP BY
        shop,
        country,
        language
    ORDER BY
        shop,
        product_rows DESC;
""").pl()

product_market_combinations

shop,country,language,product_rows
str,str,str,i64
"""frc""","""be""","""nl""",6327
"""gc""","""be""","""nl""",3072
"""gc""","""lu""","""nl""",2731
"""ha""","""nl""","""nl""",8069
"""ha""","""be""","""nl""",1820
…,…,…,…
"""idla""","""lu""","""fr""",434
"""ldil""","""nl""","""nl""",124
"""ldil""","""be""","""nl""",102


#### Findings — Context

Several shops span multiple countries; `idla` also has multiple languages. Preserve explicit country/language and do not infer either from shop.


### 10.4 Conclusion — Products

**Established:** 32,826 rows; unique `daltix_id`. Several retailers span multiple countries, with multiple languages for `idla`.

**Caveats:** semantic missingness: description **24.41%**, brand **3.90%**, categories **1.51%**, name **0.13%**. A shared ID format does not validate a join to weekly products.

**Silver plan at discovery:** use `daltix_id` as source business key; normalize assessed semantic missing values and keep descriptions nullable. Retain country/language explicitly. Evaluate this as an enrichment source only after relationship and coverage checks.


## 11. Locations

### 11.1 Grain & Uniqueness

Test `id`, `(shop, id)` and country context; inspect repeated keys rather than choosing a record arbitrarily.


In [50]:
# Check how location uniqueness changes when retailer and country context are added.
# This helps determine the likely grain and candidate key for the locations sample.

locations_path = (RAW_DIR / "locations.parquet").as_posix()

locations_grain = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT id)
            AS unique_location_ids,

        COUNT(DISTINCT (shop, id))
            AS unique_shop_location_pairs,

        COUNT(DISTINCT (shop, country_code, id))
            AS unique_location_contexts

    FROM read_parquet('{locations_path}');
""").pl()

locations_grain

total_rows,unique_location_ids,unique_shop_location_pairs,unique_location_contexts
i64,i64,i64,i64
1638,1256,1637,1637


In [51]:
# Inspect the single retailer-location combination that appears more than once.
# This determines whether it is an exact duplicate or two conflicting location records.

duplicated_location_key = duck.sql(f"""
    SELECT
        *
    FROM read_parquet('{locations_path}')
    WHERE (shop, id) IN (
        SELECT
            shop,
            id
        FROM read_parquet('{locations_path}')
        GROUP BY shop, id
        HAVING COUNT(*) > 1
    )
    ORDER BY shop, id;
""").pl()

duplicated_location_key

shop,country_code,id,type,geolocation_latitude,geolocation_longitude,postcode,sources
str,str,str,str,f64,f64,str,str
"""lld""","""be""","""f334d""",null,50.601871,4.4583208,"""1470""","""[ ""online"" ]"""
"""lld""","""be""","""f334d""",null,50.452781,4.1538596,"""7100""","""[ ""online"" ]"""


#### Findings — Grain

**1,638 rows, 1,637 `(shop, id)` combinations.** `lld + f334d` has two records with different postcodes and coordinates; adding country does not resolve it. This is a key collision, not an exact duplicate.


### 11.2 Missingness

Check optional attributes, including the entirely empty `type` field.


In [52]:
# Check missing values across the main location attributes.
# Text fields are checked for both SQL NULLs and blank values.

locations_missingness = duck.sql(f"""
    SELECT
        'country_code' AS field,
        COUNT(*) FILTER (
            WHERE country_code IS NULL
               OR TRIM(country_code) = ''
        ) AS missing_rows
    FROM read_parquet('{locations_path}')

    UNION ALL

    SELECT
        'type',
        COUNT(*) FILTER (
            WHERE type IS NULL
               OR TRIM(type) = ''
        )
    FROM read_parquet('{locations_path}')

    UNION ALL

    SELECT
        'latitude',
        COUNT(*) FILTER (
            WHERE geolocation_latitude IS NULL
        )
    FROM read_parquet('{locations_path}')

    UNION ALL

    SELECT
        'longitude',
        COUNT(*) FILTER (
            WHERE geolocation_longitude IS NULL
        )
    FROM read_parquet('{locations_path}')

    UNION ALL

    SELECT
        'postcode',
        COUNT(*) FILTER (
            WHERE postcode IS NULL
               OR TRIM(postcode) = ''
        )
    FROM read_parquet('{locations_path}')

    UNION ALL

    SELECT
        'sources',
        COUNT(*) FILTER (
            WHERE sources IS NULL
               OR TRIM(sources) = ''
        )
    FROM read_parquet('{locations_path}')
""").pl()

locations_missingness = locations_missingness.with_columns(
    (pl.col("missing_rows") / locations_grain.item(0, "total_rows") * 100)
    .round(2)
    .alias("missing_pct")
)

locations_missingness.sort("missing_pct", descending=True)

field,missing_rows,missing_pct
str,i64,f64
"""type""",1638,100.0
"""postcode""",40,2.44
"""latitude""",18,1.1
"""longitude""",18,1.1
"""country_code""",0,0.0
"""sources""",0,0.0


#### Findings — Missingness

`type`: **100%**; postcode: **2.44%**; latitude/longitude: **1.10%** each. Country and sources are populated. Consolidated Silver decisions follow at the end of this table.


### 11.3 Country Consistency

Inspect retailer/country coverage before assuming one market per shop.


In [53]:
# Check how many countries are represented within each retailer.
# This determines whether shop can safely be interpreted as a single-market identifier.

location_country_consistency = duck.sql(f"""
    SELECT
        shop,
        COUNT(*) AS location_rows,
        COUNT(DISTINCT country_code) AS distinct_countries
    FROM read_parquet('{locations_path}')
    GROUP BY shop
    ORDER BY shop;
""").pl()

location_country_consistency

shop,location_rows,distinct_countries
str,i64,i64
"""frc""",47,1
"""gc""",41,2
"""ha""",55,2
"""idla""",392,4
"""ldil""",302,3
"""lld""",749,1
"""obmuj""",9,2
"""plc""",42,1
"""raps""",1,1


In [54]:
# Inspect the actual country coverage of each retailer.

location_country_combinations = duck.sql(f"""
    SELECT
        shop,
        country_code,
        COUNT(*) AS location_rows
    FROM read_parquet('{locations_path}')
    GROUP BY
        shop,
        country_code
    ORDER BY
        shop,
        location_rows DESC;
""").pl()

location_country_combinations

shop,country_code,location_rows
str,str,i64
"""frc""","""be""",47
"""gc""","""be""",40
"""gc""","""lu""",1
"""ha""","""be""",54
"""ha""","""nl""",1
…,…,…
"""lld""","""be""",749
"""obmuj""","""nl""",6
"""obmuj""","""be""",3


#### Findings — Country

Several retailers span multiple countries. Keep rare combinations for validation rather than classifying rarity as an error; do not derive country from shop.


### 11.4 Geographic Plausibility

Check coordinate bounds and half-missing pairs. These tests establish technical validity, not verified addresses.


In [55]:
# Check whether geographic coordinates are technically valid and complete.
# This identifies impossible coordinate values and incomplete latitude/longitude pairs.

locations_geographic_validity = duck.sql(f"""
    SELECT
        COUNT(*) AS total_locations,

        COUNT(*) FILTER (
            WHERE geolocation_latitude IS NOT NULL
              AND (geolocation_latitude < -90 OR geolocation_latitude > 90)
        ) AS invalid_latitudes,

        COUNT(*) FILTER (
            WHERE geolocation_longitude IS NOT NULL
              AND (geolocation_longitude < -180 OR geolocation_longitude > 180)
        ) AS invalid_longitudes,

        COUNT(*) FILTER (
            WHERE
                (geolocation_latitude IS NULL AND geolocation_longitude IS NOT NULL)
                OR
                (geolocation_latitude IS NOT NULL AND geolocation_longitude IS NULL)
        ) AS incomplete_coordinate_pairs,

        MIN(geolocation_latitude) AS minimum_latitude,
        MAX(geolocation_latitude) AS maximum_latitude,
        MIN(geolocation_longitude) AS minimum_longitude,
        MAX(geolocation_longitude) AS maximum_longitude

    FROM read_parquet('{locations_path}');
""").pl()

locations_geographic_validity

total_locations,invalid_latitudes,invalid_longitudes,incomplete_coordinate_pairs,minimum_latitude,maximum_latitude,minimum_longitude,maximum_longitude
i64,i64,i64,i64,f64,f64,f64,f64
1638,0,0,0,49.559412,59.713997,2.5924207,14.169844


#### Findings — Geography

No out-of-range coordinates or half-missing pairs. Keep rows with absent coordinates and retain extremes for contextual validation; do not apply one country's geographic limits globally.


### 11.5 Conclusion — Locations

**Established:** 1,638 records; 1,637 `(shop, id)` combinations. Populated coordinates pass range/pair checks; multiple countries are represented.

**Caveats:** `lld + f334d` has conflicting postcode/coordinate records. `type` is entirely null; postcode is **2.44% missing**, coordinates **1.10%** each.

**Silver plan at discovery:** retain `(shop, id)` as a working key and flag the collision for explicit resolution. Omit `type` from Silver for this snapshot; retain it unchanged in Raw. Keep country and sources, nullable geography/postcodes and rare country combinations. Do not infer country from shop.


## 12. Nutritionals

### 12.1 Grain & Uniqueness

Test product, retailer, country and collection date to distinguish product identity from repeated observations over time.


In [56]:
# Check how row uniqueness changes when retailer, country and collection date are added.
# This helps determine the likely grain of the nutritional dataset.

nutritionals_path = (RAW_DIR / "nutritionals.parquet").as_posix()

nutritionals_grain = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT daltix_id)
            AS unique_daltix_ids,

        COUNT(DISTINCT (daltix_id, shop))
            AS unique_product_shop_pairs,

        COUNT(DISTINCT (daltix_id, shop, country))
            AS unique_product_contexts,

        COUNT(DISTINCT (daltix_id, shop, country, download_date))
            AS unique_candidate_grain

    FROM read_parquet('{nutritionals_path}');
""").pl()

nutritionals_grain

total_rows,unique_daltix_ids,unique_product_shop_pairs,unique_product_contexts,unique_candidate_grain
i64,i64,i64,i64,i64
1096542,54155,54155,54155,1096442


### 12.2 Repeated Observations

Inspect `(daltix_id, shop, country, download_date)` repeats, comparing serialized payload and language.


In [57]:
# Check repeated nutritional observations at the proposed grain.
# Distinguish identical repeated records from observations with different nutritional values.

nutritionals_grain_issues = duck.sql(f"""
    WITH grain AS (
        SELECT
            daltix_id,
            shop,
            country,
            download_date,

            COUNT(*) AS rows_per_grain,

            COUNT(
                DISTINCT (
                    nutritional_values_std,
                    language
                )
            ) AS distinct_versions

        FROM read_parquet('{nutritionals_path}')
        GROUP BY ALL
    )

    SELECT
        COUNT(*) AS repeated_grain_combinations,

        COUNT(*) FILTER (
            WHERE distinct_versions = 1
        ) AS identical_repeated_combinations,

        COUNT(*) FILTER (
            WHERE distinct_versions > 1
        ) AS same_grain_different_values

    FROM grain
    WHERE rows_per_grain > 1;
""").pl()

nutritionals_grain_issues

repeated_grain_combinations,identical_repeated_combinations,same_grain_different_values
i64,i64,i64
100,0,100


#### Findings — Initial Repeat

There are **100 repeated keys**, with no identical payload/language versions. Inspect details and separate language differences from payload differences next.


In [58]:
# Inspect repeated product-date observations with different nutritional values.
# This helps determine what actually differs between the two versions.

repeated_nutritional_details = duck.sql(f"""
    SELECT
        daltix_id,
        shop,
        country,
        download_date,
        language,
        nutritional_values_std
    FROM read_parquet('{nutritionals_path}')
    WHERE (daltix_id, shop, country, download_date) IN (
        SELECT
            daltix_id,
            shop,
            country,
            download_date
        FROM read_parquet('{nutritionals_path}')
        GROUP BY
            daltix_id,
            shop,
            country,
            download_date
        HAVING COUNT(*) > 1
    )
    ORDER BY
        daltix_id,
        download_date;
""").pl()

repeated_nutritional_details

daltix_id,shop,country,download_date,language,nutritional_values_std
str,str,str,date,str,str
"""05766389442155ce47fa08689d547c…","""ha""","""nl""",2020-12-18,"""nl""","""{ ""nutrients"": { ""carboh…"
"""05766389442155ce47fa08689d547c…","""ha""","""nl""",2020-12-18,"""nl""","""{ ""nutrients"": { ""carboh…"
"""08e74e5c36f50fde38757429acf3bb…","""frc""","""be""",2021-02-02,"""nl""","""{ ""nutrients"": { ""carboh…"
"""08e74e5c36f50fde38757429acf3bb…","""frc""","""be""",2021-02-02,"""nl""","""{ ""nutrients"": { ""carboh…"
"""17c65f953f988a273400c335cd7e73…","""ha""","""nl""",2020-12-29,"""nl""","""{ ""nutrients"": { ""carboh…"
…,…,…,…,…,…
"""f265a28ade5f5c940ff145f04005cc…","""ha""","""nl""",2020-12-18,"""nl""","""{ ""nutrients"": { ""carboh…"
"""f58c682d610b9b2a68626add95aabc…","""frc""","""be""",2021-02-09,"""nl""","""{ ""nutrients"": { ""carboh…"
"""f58c682d610b9b2a68626add95aabc…","""frc""","""be""",2021-02-09,"""nl""","""{ ""nutrients"": { ""carboh…"


In [59]:
# Check whether repeated product-date records differ by language,
# nutritional values, or both.

nutritionals_conflict_type = duck.sql(f"""
    WITH repeated AS (
        SELECT
            daltix_id,
            shop,
            country,
            download_date
        FROM read_parquet('{nutritionals_path}')
        GROUP BY
            daltix_id,
            shop,
            country,
            download_date
        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS repeated_grains,

        COUNT(*) FILTER (
            WHERE distinct_languages > 1
        ) AS language_conflicts,

        COUNT(*) FILTER (
            WHERE distinct_nutritional_values > 1
        ) AS nutritional_value_conflicts

    FROM (
        SELECT
            n.daltix_id,
            n.shop,
            n.country,
            n.download_date,
            COUNT(DISTINCT n.language) AS distinct_languages,
            COUNT(DISTINCT n.nutritional_values_std) AS distinct_nutritional_values
        FROM read_parquet('{nutritionals_path}') n
        INNER JOIN repeated r
            ON n.daltix_id = r.daltix_id
           AND n.shop = r.shop
           AND n.country = r.country
           AND n.download_date = r.download_date
        GROUP BY
            n.daltix_id,
            n.shop,
            n.country,
            n.download_date
    );
""").pl()

nutritionals_conflict_type

repeated_grains,language_conflicts,nutritional_value_conflicts
i64,i64,i64
100,0,100


#### Findings — Conflict

All **100 repeated keys have two rows** with the same language and different serialized nutritional payloads. Retain both versions pending resolution; this comparison does not itself establish which nutrient values differ.


### 12.3 Temporal Coverage

Check the collection-date range at dataset level.


In [60]:
# Describe the overall time range and collection-date coverage
# of the nutritional dataset.

nutritionals_temporal_coverage = duck.sql(f"""
    SELECT
        MIN(download_date) AS first_date,
        MAX(download_date) AS last_date,
        COUNT(DISTINCT download_date) AS observed_dates,
        DATE_DIFF(
            'day',
            MIN(download_date),
            MAX(download_date)
        ) + 1 AS calendar_days
    FROM read_parquet('{nutritionals_path}');
""").pl()

nutritionals_temporal_coverage

first_date,last_date,observed_dates,calendar_days
date,date,i64,i64
2020-11-27,2021-02-24,90,90


#### Findings — Coverage

**2020-11-27 → 2021-02-24: 90/90 dates.** Date coverage is complete across the dataset, not necessarily for each product.


### 12.4 Missingness

Check core identifiers, date, language and payload for nulls/blank text. Populated JSON text does not prove complete nutritional content.


In [61]:
# Check missing values across the main nutritional fields.
# Text fields are checked for both SQL NULLs and blank values.

nutritionals_missingness = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE daltix_id IS NULL
               OR TRIM(daltix_id) = ''
        ) AS missing_daltix_id,

        COUNT(*) FILTER (
            WHERE shop IS NULL
               OR TRIM(shop) = ''
        ) AS missing_shop,

        COUNT(*) FILTER (
            WHERE country IS NULL
               OR TRIM(country) = ''
        ) AS missing_country,

        COUNT(*) FILTER (
            WHERE download_date IS NULL
        ) AS missing_download_date,

        COUNT(*) FILTER (
            WHERE nutritional_values_std IS NULL
               OR TRIM(nutritional_values_std) = ''
        ) AS missing_nutritional_values,

        COUNT(*) FILTER (
            WHERE language IS NULL
               OR TRIM(language) = ''
        ) AS missing_language

    FROM read_parquet('{nutritionals_path}');
""").pl()

nutritionals_missingness

total_rows,missing_daltix_id,missing_shop,missing_country,missing_download_date,missing_nutritional_values,missing_language
i64,i64,i64,i64,i64,i64,i64
1096542,0,0,0,0,0,0


#### Findings — Completeness

No null/blank core fields under the checks used. Nutrient-level completeness remains separate from this structural check.


### 12.5 Payload Structure

Test JSON parseability, then compare sorted keys under `$.nutrients`.


In [62]:
# Check whether nutritional_values_std contains consistently parseable JSON-like content.
# This helps determine whether the field can be normalized safely in the Silver layer.

nutritionals_structure = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE TRY_CAST(nutritional_values_std AS JSON) IS NOT NULL
        ) AS parseable_json_rows,

        COUNT(*) FILTER (
            WHERE TRY_CAST(nutritional_values_std AS JSON) IS NULL
        ) AS non_parseable_json_rows

    FROM read_parquet('{nutritionals_path}');
""").pl()

nutritionals_structure

total_rows,parseable_json_rows,non_parseable_json_rows
i64,i64,i64
1096542,1096542,0


In [63]:
# Check how many different nutrient-key structures exist in the JSON payload.
# This determines whether nutritionals can be flattened into stable columns
# or require a more flexible normalized structure.

nutritionals_schema_consistency = duck.sql(f"""
    WITH nutrient_schemas AS (
        SELECT
            CAST(
                list_sort(
                    json_keys(
                        CAST(nutritional_values_std AS JSON),
                        '$.nutrients'
                    )
                ) AS VARCHAR
            ) AS nutrient_schema
        FROM read_parquet('{nutritionals_path}')
    )

    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT nutrient_schema) AS distinct_nutrient_schemas
    FROM nutrient_schemas;
""").pl()

nutritionals_schema_consistency

total_rows,distinct_nutrient_schemas
i64,i64
1096542,185


#### Findings — Payload

All **1,096,542 payloads parse as JSON**; the query finds **185 distinct nutrient-key sets**. Parseability is established, but common value types, units and nutrient completeness are not established by these checks.


### 12.6 Conclusion — Nutritionals

**Established:** 1,096,542 observations for 54,155 products; working grain `(daltix_id, shop, country, download_date)`; **90/90 dates**, no null/blank core fields and all JSON parseable.

**Caveats:** **100 keys** occur twice with the same language and differing serialized payloads. There are **185 nutrient-key sets**, so fields are heterogeneous; valid JSON is not proof of consistent nutrient meaning.

**Silver plan at discovery:** preserve the working grain and both conflicting versions; flag them instead of arbitrarily deduplicating. Parse nutritional data while retaining original JSON for lineage. Prefer a flexible nutrient representation with value/unit/language; do not assume one fixed wide schema or discard rows for missing core values when none were found.


**Implemented follow-up:** see [02 · Silver Pipeline](02_silver_pipeline.ipynb) for the adopted promotion convention, deterministic nutritional selection and persisted provenance. These decisions extend the discovery evidence; Raw results below remain unchanged.


## 13. Cross-Table Relationships

Check key context, coverage, cardinality and reverse coverage for each relationship. Results apply to the current snapshot; matching IDs do not establish historical attribute validity.

### 13.1 Weekly Prices ↔ Weekly Products

#### Join Key Validation

Check retailer agreement for matched `daltix_id` values; the product source has one row per ID.


In [64]:
# Check whether matched daltix_id values refer to the same retailer
# in weekly_prices and weekly_prices_products.

weekly_product_join_key = duck.sql(f"""
    SELECT
        COUNT(*) AS matched_price_rows,

        COUNT(*) FILTER (
            WHERE wp.shop = p.shop
        ) AS same_shop_rows,

        COUNT(*) FILTER (
            WHERE wp.shop <> p.shop
        ) AS different_shop_rows

    FROM read_parquet('{weekly_prices_path}') wp

    INNER JOIN read_parquet('{weekly_products_path}') p
        ON wp.daltix_id = p.daltix_id;
""").pl()

weekly_product_join_key

matched_price_rows,same_shop_rows,different_shop_rows
i64,i64,i64
18460902,18460902,0


#### Match Coverage — Products

Measure distinct product coverage so frequently observed products do not receive extra weight.


In [65]:
# Measure how much of weekly_prices is covered by weekly_prices_products.
# This is calculated at the daltix_id level to avoid weighting frequently observed products more heavily.

weekly_product_coverage = duck.sql(f"""
    SELECT
        COUNT(DISTINCT wp.daltix_id) AS price_product_ids,

        COUNT(DISTINCT p.daltix_id) AS matched_product_ids,

        COUNT(DISTINCT wp.daltix_id)
            - COUNT(DISTINCT p.daltix_id) AS unmatched_product_ids,

        ROUND(
            100.0
            * COUNT(DISTINCT p.daltix_id)
            / COUNT(DISTINCT wp.daltix_id),
            2
        ) AS product_match_pct

    FROM read_parquet('{weekly_prices_path}') wp

    LEFT JOIN read_parquet('{weekly_products_path}') p
        ON wp.daltix_id = p.daltix_id;
""").pl()

weekly_product_coverage

price_product_ids,matched_product_ids,unmatched_product_ids,product_match_pct
i64,i64,i64,f64
102069,88897,13172,87.1


##### Findings

**88,897 / 102,069 products match (87.10%)**; 13,172 lack product metadata. Coverage is incomplete.


#### Match Coverage — Price Rows

Measure how much of the observed price history has product metadata.


In [67]:
# Measure how many weekly price rows have a matching product record.
# This complements the product-level match rate with row-level coverage.

weekly_product_row_coverage = duck.sql(f"""
    SELECT
        COUNT(*) AS total_price_rows,

        COUNT(*) FILTER (
            WHERE p.daltix_id IS NOT NULL
        ) AS matched_price_rows,

        COUNT(*) FILTER (
            WHERE p.daltix_id IS NULL
        ) AS unmatched_price_rows,

        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE p.daltix_id IS NOT NULL)
            / COUNT(*),
            2
        ) AS row_match_pct

    FROM read_parquet('{weekly_prices_path}') wp

    LEFT JOIN read_parquet('{weekly_products_path}') p
        ON wp.daltix_id = p.daltix_id;
""").pl()

weekly_product_row_coverage

total_price_rows,matched_price_rows,unmatched_price_rows,row_match_pct
i64,i64,i64,f64
19218071,18460902,757169,96.06


##### Findings

**18,460,902 / 19,218,071 rows match (96.06%)**. Unmatched products account for **3.94%** of observations.


#### Cardinality

Compare source and left-join row counts to test the expected many-to-one relationship.


In [66]:
# Confirm that joining product attributes does not multiply weekly price rows.
# A safe many-to-one join should return exactly the same number of rows as weekly_prices.

weekly_product_row_explosion = duck.sql(f"""
    WITH source AS (
        SELECT COUNT(*) AS source_rows
        FROM read_parquet('{weekly_prices_path}')
    ),

    joined AS (
        SELECT COUNT(*) AS joined_rows
        FROM read_parquet('{weekly_prices_path}') wp

        LEFT JOIN read_parquet('{weekly_products_path}') p
            ON wp.daltix_id = p.daltix_id
    )

    SELECT
        source_rows,
        joined_rows,
        joined_rows - source_rows AS extra_rows_created

    FROM source, joined;
""").pl()

weekly_product_row_explosion

source_rows,joined_rows,extra_rows_created
i64,i64,i64
19218071,19218071,0


##### Findings

The left join preserves **19,218,071 rows**, with zero extra rows. Product enrichment does not multiply facts in this snapshot.


#### Reverse Coverage

Count product-reference records used by the weekly price history.


In [68]:
# Check how many weekly product records are actually used by weekly_prices.
# Product records with no matching price observation remain valid dimension records,
# but are not part of the observed weekly pricing history.

weekly_products_usage = duck.sql(f"""
    SELECT
        COUNT(*) AS total_product_rows,

        COUNT(*) FILTER (
            WHERE wp.daltix_id IS NOT NULL
        ) AS products_used_in_prices,

        COUNT(*) FILTER (
            WHERE wp.daltix_id IS NULL
        ) AS products_not_used_in_prices

    FROM read_parquet('{weekly_products_path}') p

    LEFT JOIN (
        SELECT DISTINCT daltix_id
        FROM read_parquet('{weekly_prices_path}')
    ) wp
        ON p.daltix_id = wp.daltix_id;
""").pl()

weekly_products_usage

total_product_rows,products_used_in_prices,products_not_used_in_prices
i64,i64,i64
114517,88897,25620


#### Conclusion

**Established:** matched IDs agree on retailer; the join is many-to-one and preserves fact rows. Coverage is **87.10% of products / 96.06% of price rows**.

**Caveats:** metadata is incomplete; 25,620 product-reference records are unused by this price history. Cardinality safety does not prove temporal compatibility.

**Silver plan at discovery:** retain the weekly product reference and preserve unmatched fact observations when enriching; do not treat missing metadata as grounds for dropping prices.


### 13.2 Weekly Prices ↔ Weekly Locations

#### Join Key Validation & Match Coverage

Use `(shop, location)`, already unique in the weekly reference, to measure price-row coverage.


In [69]:
# Check how many weekly price rows find a matching shop + location record.

weekly_location_join_coverage = duck.sql(f"""
    SELECT
        COUNT(*) AS total_price_rows,

        COUNT(*) FILTER (
            WHERE l.location IS NOT NULL
        ) AS matched_price_rows,

        COUNT(*) FILTER (
            WHERE l.location IS NULL
        ) AS unmatched_price_rows,

        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE l.location IS NOT NULL)
            / COUNT(*),
            2
        ) AS row_match_pct

    FROM read_parquet('{weekly_prices_path}') wp

    LEFT JOIN read_parquet('{weekly_locations_path}') l
        ON wp.shop = l.shop
       AND wp.location = l.location;
""").pl()

weekly_location_join_coverage

total_price_rows,matched_price_rows,unmatched_price_rows,row_match_pct
i64,i64,i64,f64
19218071,19218071,0,100.0


##### Findings

Every weekly price observation has a matching `(shop, location)` record: **100% row coverage**.


#### Cardinality

Compare source and left-join counts. From prices to the location reference, the expected relationship is many-to-one.


In [70]:
# Confirm that joining location attributes does not multiply weekly price rows.
# A safe many-to-one relationship should preserve the weekly_prices row count exactly.

weekly_location_row_explosion = duck.sql(f"""
    WITH source AS (
        SELECT COUNT(*) AS source_rows
        FROM read_parquet('{weekly_prices_path}')
    ),

    joined AS (
        SELECT COUNT(*) AS joined_rows
        FROM read_parquet('{weekly_prices_path}') wp

        LEFT JOIN read_parquet('{weekly_locations_path}') l
            ON wp.shop = l.shop
           AND wp.location = l.location
    )

    SELECT
        source_rows,
        joined_rows,
        joined_rows - source_rows AS extra_rows_created

    FROM source, joined;
""").pl()

weekly_location_row_explosion

source_rows,joined_rows,extra_rows_created
i64,i64,i64
19218071,19218071,0


##### Findings

The join preserves the original weekly price row count with **zero extra rows**. The reverse relationship, locations to prices, is one-to-many.


#### Reverse Coverage

Count location-reference records used by the weekly price history.


In [71]:
# Check how many weekly location records are actually used by weekly_prices.

weekly_locations_usage = duck.sql(f"""
    SELECT
        COUNT(*) AS total_location_rows,

        COUNT(*) FILTER (
            WHERE wp.location IS NOT NULL
        ) AS locations_used_in_prices,

        COUNT(*) FILTER (
            WHERE wp.location IS NULL
        ) AS locations_not_used_in_prices

    FROM read_parquet('{weekly_locations_path}') l

    LEFT JOIN (
        SELECT DISTINCT
            shop,
            location
        FROM read_parquet('{weekly_prices_path}')
    ) wp
        ON l.shop = wp.shop
       AND l.location = wp.location;
""").pl()

weekly_locations_usage

total_location_rows,locations_used_in_prices,locations_not_used_in_prices
i64,i64,i64
1230,16,1214


##### Findings

Only **16 / 1,230** reference keys are used; **1,214** are not observed in this price history. Unused does not mean invalid.


#### Conclusion

**Established:** `(shop, location)` gives **100% price-row coverage** without multiplying rows.

**Caveats:** the history uses only **16 of 1,230** reference locations; reference-table breadth is not fact-table coverage.

**Silver plan at discovery:** retain the composite key and broader reference. Keep geographic and descriptive attributes separate from business identity.


### 13.3 Prices ↔ Products

#### Join Key Validation

Check retailer/country agreement for matched `daltix_id` values; the product source has one row per ID.


In [72]:
# Check whether matched daltix_id values refer to the same shop and country.
# This validates daltix_id as a semantically consistent join key between prices and products.

prices_product_join_key = duck.sql(f"""
    SELECT
        COUNT(*) AS matched_price_rows,

        COUNT(*) FILTER (
            WHERE pr.shop = p.shop
              AND pr.country = p.country
        ) AS same_context_rows,

        COUNT(*) FILTER (
            WHERE pr.shop <> p.shop
               OR pr.country <> p.country
        ) AS different_context_rows

    FROM read_parquet('{prices_path}') pr

    INNER JOIN read_parquet('{products_path}') p
        ON pr.daltix_id = p.daltix_id;
""").pl()

prices_product_join_key

matched_price_rows,same_context_rows,different_context_rows
i64,i64,i64
184580,184580,0


##### Findings

All matched records agree on **shop and country**. No context conflicts were found.


#### Match Coverage — Products

Measure distinct product coverage in the historical price sample.


In [73]:
# Measure how many unique products in prices have a matching record in products.
# This shows the product-level metadata coverage of the historical price sample.

prices_product_coverage = duck.sql(f"""
    SELECT
        COUNT(DISTINCT pr.daltix_id) AS price_product_ids,

        COUNT(DISTINCT p.daltix_id) AS matched_product_ids,

        COUNT(DISTINCT pr.daltix_id)
            - COUNT(DISTINCT p.daltix_id) AS unmatched_product_ids,

        ROUND(
            100.0
            * COUNT(DISTINCT p.daltix_id)
            / COUNT(DISTINCT pr.daltix_id),
            2
        ) AS product_match_pct

    FROM read_parquet('{prices_path}') pr

    LEFT JOIN read_parquet('{products_path}') p
        ON pr.daltix_id = p.daltix_id;
""").pl()

prices_product_coverage

price_product_ids,matched_product_ids,unmatched_product_ids,product_match_pct
i64,i64,i64,f64
116,20,96,17.24


#### Match Coverage — Price Rows

Measure the share of historical price observations with product metadata.


In [74]:
# Measure how many historical price rows have matching product metadata.
# This complements the unique-product match rate with row-level coverage.

prices_product_row_coverage = duck.sql(f"""
    SELECT
        COUNT(*) AS total_price_rows,

        COUNT(*) FILTER (
            WHERE p.daltix_id IS NOT NULL
        ) AS matched_price_rows,

        COUNT(*) FILTER (
            WHERE p.daltix_id IS NULL
        ) AS unmatched_price_rows,

        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE p.daltix_id IS NOT NULL)
            / COUNT(*),
            2
        ) AS row_match_pct

    FROM read_parquet('{prices_path}') pr

    LEFT JOIN read_parquet('{products_path}') p
        ON pr.daltix_id = p.daltix_id;
""").pl()

prices_product_row_coverage

total_price_rows,matched_price_rows,unmatched_price_rows,row_match_pct
i64,i64,i64,f64
1198547,184580,1013967,15.4


##### Findings

Only **20 / 116 products match (17.24%)**, covering **15.40% of price rows**. Metadata coverage is limited.


#### Cardinality

Check that the left join preserves price rows despite incomplete matches.


In [75]:
# Confirm that joining products does not multiply historical price rows.
# Limited coverage is acceptable, but the matched join must still preserve row count.

prices_product_row_explosion = duck.sql(f"""
    WITH source AS (
        SELECT COUNT(*) AS source_rows
        FROM read_parquet('{prices_path}')
    ),

    joined AS (
        SELECT COUNT(*) AS joined_rows
        FROM read_parquet('{prices_path}') pr

        LEFT JOIN read_parquet('{products_path}') p
            ON pr.daltix_id = p.daltix_id
    )

    SELECT
        source_rows,
        joined_rows,
        joined_rows - source_rows AS extra_rows_created

    FROM source, joined;
""").pl()

prices_product_row_explosion

source_rows,joined_rows,extra_rows_created
i64,i64,i64
1198547,1198547,0


##### Findings

The many-to-one join preserves **1,198,547 rows**, with **zero extra rows**.


#### Reverse Coverage

Count records in the broader product reference that occur in `prices`.


In [76]:
# Check how many product reference records are actually used by prices.
# This shows how much of the broader products table participates in the historical pricing sample.

products_usage_in_prices = duck.sql(f"""
    SELECT
        COUNT(*) AS total_product_rows,

        COUNT(*) FILTER (
            WHERE pr.daltix_id IS NOT NULL
        ) AS products_used_in_prices,

        COUNT(*) FILTER (
            WHERE pr.daltix_id IS NULL
        ) AS products_not_used_in_prices

    FROM read_parquet('{products_path}') p

    LEFT JOIN (
        SELECT DISTINCT daltix_id
        FROM read_parquet('{prices_path}')
    ) pr
        ON p.daltix_id = pr.daltix_id;
""").pl()

products_usage_in_prices

total_product_rows,products_used_in_prices,products_not_used_in_prices
i64,i64,i64
32826,20,32806


##### Findings

Only **20 / 32,826** product-reference records occur in the historical price sample.


#### Conclusion

**Established:** matched IDs agree on shop/country; the many-to-one join preserves price rows.

**Caveats:** coverage is **17.24% of products / 15.40% of price rows**; only 20 reference records participate.

**Silver plan at discovery:** retain `products` as optional metadata for matched historical prices. Preserve the price observations whose metadata is absent.


### 13.4 Prices ↔ Locations

#### Join Key Validation & Match Coverage

Compare `location = id`, then add shop, then country. `EXISTS` measures coverage without inflating counts when reference keys repeat.


In [77]:
# Compare possible location join keys between prices and locations.
# EXISTS is used so that duplicated location records cannot inflate the price row count.

prices_location_join_key = duck.sql(f"""
    SELECT
        COUNT(*) AS total_price_rows,

        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1
                FROM read_parquet('{locations_path}') l
                WHERE pr.location = l.id
            )
        ) AS matched_on_location_only,

        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1
                FROM read_parquet('{locations_path}') l
                WHERE pr.location = l.id
                  AND pr.shop = l.shop
            )
        ) AS matched_on_shop_location,

        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1
                FROM read_parquet('{locations_path}') l
                WHERE pr.location = l.id
                  AND pr.shop = l.shop
                  AND pr.country = l.country_code
            )
        ) AS matched_on_full_context

    FROM read_parquet('{prices_path}') pr;
""").pl()

prices_location_join_key

total_price_rows,matched_on_location_only,matched_on_shop_location,matched_on_full_context
i64,i64,i64,i64
1198547,1198547,1198547,1198547


##### Findings

All **1,198,547 price rows** match under each candidate key, including full shop/country context. Coverage alone cannot select a safe join key.


#### Cardinality

Compare left-join row counts for the three candidate keys.


In [78]:
# Compare row counts produced by each candidate location join key.
# The safest key should preserve the original prices row count without creating extra rows.

prices_location_cardinality = duck.sql(f"""
    WITH source AS (
        SELECT COUNT(*) AS source_rows
        FROM read_parquet('{prices_path}')
    ),

    location_only AS (
        SELECT COUNT(*) AS joined_rows
        FROM read_parquet('{prices_path}') pr
        LEFT JOIN read_parquet('{locations_path}') l
            ON pr.location = l.id
    ),

    shop_location AS (
        SELECT COUNT(*) AS joined_rows
        FROM read_parquet('{prices_path}') pr
        LEFT JOIN read_parquet('{locations_path}') l
            ON pr.shop = l.shop
           AND pr.location = l.id
    ),

    full_context AS (
        SELECT COUNT(*) AS joined_rows
        FROM read_parquet('{prices_path}') pr
        LEFT JOIN read_parquet('{locations_path}') l
            ON pr.shop = l.shop
           AND pr.country = l.country_code
           AND pr.location = l.id
    )

    SELECT
        source.source_rows,

        location_only.joined_rows - source.source_rows
            AS extra_rows_location_only,

        shop_location.joined_rows - source.source_rows
            AS extra_rows_shop_location,

        full_context.joined_rows - source.source_rows
            AS extra_rows_full_context

    FROM source, location_only, shop_location, full_context;
""").pl()

prices_location_cardinality

source_rows,extra_rows_location_only,extra_rows_shop_location,extra_rows_full_context
i64,i64,i64,i64
1198547,2253016,0,0


##### Findings

Location alone creates **2,253,016 extra rows**. Adding shop removes multiplication for this sample; adding country has no further cardinality benefit. Use `(shop, location)` against `(shop, id)`.


#### Reverse Coverage

Count distinct reference keys used by `prices`; account for the known duplicated key in `locations`.


In [79]:
# Check how many distinct location business keys are actually used by prices.
# Distinct shop + id is used because locations contains one known key collision.

prices_location_reverse_coverage = duck.sql(f"""
    WITH location_keys AS (
        SELECT DISTINCT
            shop,
            id
        FROM read_parquet('{locations_path}')
    ),

    price_location_keys AS (
        SELECT DISTINCT
            shop,
            location
        FROM read_parquet('{prices_path}')
    )

    SELECT
        COUNT(*) AS total_location_keys,

        COUNT(*) FILTER (
            WHERE p.location IS NOT NULL
        ) AS locations_used_in_prices,

        COUNT(*) FILTER (
            WHERE p.location IS NULL
        ) AS locations_not_used_in_prices,

        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE p.location IS NOT NULL)
            / COUNT(*),
            2
        ) AS location_usage_pct

    FROM location_keys l

    LEFT JOIN price_location_keys p
        ON l.shop = p.shop
       AND l.id = p.location;
""").pl()

prices_location_reverse_coverage

total_location_keys,locations_used_in_prices,locations_not_used_in_prices,location_usage_pct
i64,i64,i64,f64
1637,136,1501,8.31


##### Findings

Only **136 / 1,637 location keys (8.31%)** are used by the historical price sample.


#### Conclusion

**Established:** the shop/location join gives **100% coverage** and preserves price rows; location alone is unsafe.

**Caveats:** only 136 reference keys participate. Passing this join test does not resolve the source's separate `lld + f334d` collision.

**Silver plan at discovery:** keep retailer context in the join and handle the known reference collision explicitly before enforcing source-key uniqueness.


### 13.5 Weekly Products ↔ Nutritionals

#### Join Key Validation

Check shop/country agreement for nutritional observations matching weekly product IDs.


In [80]:
# Check whether matched daltix_id values refer to the same retailer and country context.
# This validates whether daltix_id can be used safely between weekly products and nutritionals.

weekly_products_nutritionals_join_key = duck.sql(f"""
    SELECT
        COUNT(*) AS matched_nutritional_rows,

        COUNT(*) FILTER (
            WHERE wp.shop = n.shop
              AND wp.country = n.country
        ) AS same_context_rows,

        COUNT(*) FILTER (
            WHERE wp.shop <> n.shop
               OR wp.country <> n.country
        ) AS different_context_rows

    FROM read_parquet('{nutritionals_path}') n

    INNER JOIN read_parquet('{weekly_products_path}') wp
        ON n.daltix_id = wp.daltix_id;
""").pl()

weekly_products_nutritionals_join_key

matched_nutritional_rows,same_context_rows,different_context_rows
i64,i64,i64
261224,261224,0


#### Match Coverage — Products

Use distinct product presence: multiple nutritional observations must not inflate product coverage.


In [81]:
# Check how many weekly products have at least one nutritional observation.
# EXISTS avoids multiplying product rows because nutritionals contains historical records.

weekly_products_nutritionals_coverage = duck.sql(f"""
    SELECT
        COUNT(*) AS total_weekly_products,

        COUNT(*) FILTER (
            WHERE EXISTS (
                SELECT 1
                FROM read_parquet('{nutritionals_path}') n
                WHERE n.daltix_id = wp.daltix_id
            )
        ) AS weekly_products_with_nutrition,

        COUNT(*) FILTER (
            WHERE NOT EXISTS (
                SELECT 1
                FROM read_parquet('{nutritionals_path}') n
                WHERE n.daltix_id = wp.daltix_id
            )
        ) AS weekly_products_without_nutrition,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE EXISTS (
                    SELECT 1
                    FROM read_parquet('{nutritionals_path}') n
                    WHERE n.daltix_id = wp.daltix_id
                )
            ) / COUNT(*),
            2
        ) AS product_coverage_pct

    FROM read_parquet('{weekly_products_path}') wp;
""").pl()

weekly_products_nutritionals_coverage

total_weekly_products,weekly_products_with_nutrition,weekly_products_without_nutrition,product_coverage_pct
i64,i64,i64,f64
114517,15696,98821,13.71


##### Findings

Nutritionals cover **15,696 / 114,517 weekly products (13.71%)**. This is optional enrichment, not a complete attribute source.


#### Cardinality

Count nutritional observations per matched product before attaching history to a one-row-per-product reference.


In [82]:
# Measure how many nutritional observations exist per matched weekly product.
# This shows whether nutritionals can be joined directly to a one-row-per-product dimension.

weekly_products_nutritionals_cardinality = duck.sql(f"""
    WITH nutrition_counts AS (
        SELECT
            daltix_id,
            COUNT(*) AS nutritional_rows
        FROM read_parquet('{nutritionals_path}')
        GROUP BY daltix_id
    ),

    matched_products AS (
        SELECT
            wp.daltix_id,
            nc.nutritional_rows
        FROM read_parquet('{weekly_products_path}') wp
        INNER JOIN nutrition_counts nc
            ON wp.daltix_id = nc.daltix_id
    )

    SELECT
        COUNT(*) AS matched_products,
        SUM(nutritional_rows) AS matched_nutritional_rows,
        ROUND(AVG(nutritional_rows), 2) AS avg_rows_per_product,
        MEDIAN(nutritional_rows) AS median_rows_per_product,
        MIN(nutritional_rows) AS min_rows_per_product,
        MAX(nutritional_rows) AS max_rows_per_product,
        COUNT(*) FILTER (
            WHERE nutritional_rows > 1
        ) AS products_with_multiple_observations

    FROM matched_products;
""").pl()

weekly_products_nutritionals_cardinality

matched_products,matched_nutritional_rows,avg_rows_per_product,median_rows_per_product,min_rows_per_product,max_rows_per_product,products_with_multiple_observations
i64,"decimal[38,0]",f64,f64,i64,i64,i64
15696,261224,16.64,19.0,1,32,14145


##### Findings

The **15,696 matched products** have **261,224 observations**: mean **16.64**, median **19**, maximum **32**. **14,145 products** have multiple observations. A direct join would break product-level uniqueness.


#### Reverse Coverage

Measure the share of distinct nutritional products represented in the weekly reference.


In [83]:
# Check how many distinct nutritional products are represented in weekly products.
# Distinct daltix_id is used because nutritionals contains multiple historical rows per product.

weekly_products_nutritionals_reverse_coverage = duck.sql(f"""
    WITH nutritional_products AS (
        SELECT DISTINCT
            daltix_id
        FROM read_parquet('{nutritionals_path}')
    ),

    weekly_products AS (
        SELECT DISTINCT
            daltix_id
        FROM read_parquet('{weekly_products_path}')
    )

    SELECT
        COUNT(*) AS total_nutritional_products,

        COUNT(*) FILTER (
            WHERE wp.daltix_id IS NOT NULL
        ) AS nutritional_products_in_weekly,

        COUNT(*) FILTER (
            WHERE wp.daltix_id IS NULL
        ) AS nutritional_products_not_in_weekly,

        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE wp.daltix_id IS NOT NULL)
            / COUNT(*),
            2
        ) AS reverse_coverage_pct

    FROM nutritional_products n

    LEFT JOIN weekly_products wp
        ON n.daltix_id = wp.daltix_id;
""").pl()

weekly_products_nutritionals_reverse_coverage

total_nutritional_products,nutritional_products_in_weekly,nutritional_products_not_in_weekly,reverse_coverage_pct
i64,i64,i64,f64
54155,15696,38459,28.98


##### Findings

**15,696 / 54,155 nutritional products match (28.98%)**; 38,459 lie outside the weekly reference. Overlap is partial in both directions.


#### Conclusion

**Established:** all **261,224 matched observations** agree on shop/country. The relationship is one-to-many, with **13.71% forward / 28.98% reverse product coverage**.

**Caveats:** repeated historical observations cannot become one static product row without a temporal or aggregation rule. Collection date is not automatically an effective date.

**Silver plan at discovery:** preserve nutritional history as optional enrichment. Keep its observation grain and original payload; avoid a direct many-row join into the product reference.


## 14. Final EDA Conclusion

**Established**

The weekly sources form the core analytical group: `weekly_prices` provides 19.2M observations across 104 weeks, with product and location references. Product metadata matches **96.06% of price rows**; location metadata matches **100%** using `(shop, location)`. Both left joins preserve the fact row count. The price history uses only **16 reference locations**, so the wider location catalogue does not describe observed geographic coverage.

Non-weekly prices remain a supporting historical sample of **116 products**. Their product metadata coverage is limited, while retailer/location joins preserve rows. Nutritionals cover **13.71% of weekly products** and contain multiple dated observations; they remain optional historical enrichment.

**Caveats**

The weekly grain still contains **518,884 repeated groups** across three weeks. Distinguish full-row exact duplicates from conflicting prices; retain Raw and apply only documented Silver rules. Extreme prices, nullable descriptions and shared coordinates are not automatic deletion criteria. The location-key collision and 100 repeated nutritional keys also require explicit handling.

Promotion conventions remain source-specific. In `prices`, a null promotional field means no promotion was recorded; it does not establish that none existed. Nutritional JSON is parseable but has **185 nutrient-key sets**, not one validated typed schema. Context matching and join cardinality do not establish historical compatibility.

**Silver plan**

Implement the per-source decisions with typed fields, normalized semantic missingness, lineage and explicit quality flags. Preserve unmatched facts, avoid arbitrary conflict resolution, and keep nutritional history separate until a temporal rule is justified. Retain country independently of retailer and protect joins against row multiplication.

After Silver validation, develop the provisional weekly fact/dimension model and focused business analyses. The [README](../README.md#delivery-sequence) retains the detailed architecture, validation scope and delivery sequence.


**Implemented follow-up:** see [02 · Silver Pipeline](02_silver_pipeline.ipynb) for the adopted promotion convention, deterministic nutritional selection and persisted provenance. These decisions extend the discovery evidence; Raw results below remain unchanged.
